# 67 · Post-E50 — Sagittal-Based Automatic Disc Instance and Level Localization

**Objetivo:** construir un pipeline reproducible que va de Sagittal T2 → segmentación existente
(`sagittal_spider`) → instancias discales individuales → orden anatómico cráneo-caudal →
(intento de) naming absoluto de niveles lumbares (`L1-L2`...`L5-S1`) con **abstención explícita**
cuando la evidencia no alcanza, separando en todo momento:

- **INSTANCE LOCALIZATION** (¿cuántos discos hay y dónde están?) de
- **ABSOLUTE LEVEL NAMING** (¿cuál es cuál?).

## Herencia de fases anteriores

- Notebook 65: inventario baseline.
- Notebook 66: geometría DICOM intraserie validada (Sagittal T1/T2/Axial T2); 5 orientation
  clusters reales en Axial T2.
- Notebook 66B: cross-frame registration baseline establecido; multiplanar **NO** validado;
  `ready_for_sagittal_level_localization = YES`.
- `AUTOMATIC_DISC_LOCALIZATION_VALIDATED` sigue en `False` en el código productivo y **no se
  modifica en este notebook** — solo se genera evidencia.

## Alcance

**SÍ:** inferencia real (frozen, read-only) con el checkpoint `sagittal_spider` sobre Sagittal T2
(y Sagittal T1 solo como evidencia secundaria); extracción de instancias discales; consenso
multi-slice; eje cráneo-caudal en patient-space; ordenamiento anatómico; naming absoluto **con
abstención** cuando falta anchor confiable; validación contra SPIDER held-out si está disponible
localmente; resultado exploratorio sobre el DICOM real (marcado `INFERRED`, nunca `VALIDATED`
sin evidencia externa).

**NO:** Axial T2, cross-frame registration, Notebook 67B, entrenamiento de ningún modelo,
modificación de código productivo ni de `AUTOMATIC_DISC_LOCALIZATION_VALIDATED`.

Rama: `research/post-e50-level-localization` · Rama madre: `research/post-e50-cross-frame-registration`


In [1]:
# --- Setup: repository root, allowed write scope, git identity, privacy helpers ---
import hashlib
import io
import json
import os
import subprocess
import sys
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch

EXECUTION_START = time.time()


def _find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "ai_service").is_dir() and (candidate / "config").is_dir():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook location")


NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = _find_repo_root(NOTEBOOK_DIR)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ALLOWED_WRITE_ROOTS = [
    REPO_ROOT / "notebooks" / "post_e50",
    REPO_ROOT / "artifacts" / "post_e50",
    REPO_ROOT / "reports" / "post_e50",
]
LOC_DIR = REPO_ROOT / "artifacts" / "post_e50" / "level_localization"
FIGURES_DIR = LOC_DIR / "figures"
REPORT_DIR = REPO_ROOT / "reports" / "post_e50"

warnings: list[str] = []
limitations: list[str] = []

FORBIDDEN_IDENTIFIER_FIELDS = (
    "PatientName", "PatientID", "AccessionNumber",
    "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "InstitutionName",
)


def safe_write_text(path: Path, content: str) -> None:
    path = path.resolve()
    if not any(str(path).startswith(str(root.resolve())) for root in ALLOWED_WRITE_ROOTS):
        raise RuntimeError(f"Refusing to write outside allowed Post-E50 trees: {path}")
    forbidden_values = getattr(safe_write_text, "_forbidden_values", set())
    for value in forbidden_values:
        if value and value in content:
            raise RuntimeError(f"Refusing to persist forbidden identifier value into {path.name}")
    if "C:\\Users\\" in content or "/Users/" in content:
        raise RuntimeError(f"Refusing to persist a local filesystem path into {path.name}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")


def opaque_id(raw_uid: str) -> str:
    return hashlib.sha256(str(raw_uid).encode("utf-8")).hexdigest()[:12]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run_git(*args: str) -> str:
    result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
    return result.stdout.strip()


GIT_BRANCH = run_git("branch", "--show-current")
GIT_COMMIT = run_git("rev-parse", "HEAD")
GIT_STATUS_PORCELAIN = run_git("status", "--porcelain")
GENERATED_AT = datetime.now(timezone.utc).isoformat()

print("REPO_ROOT (relative label only):", REPO_ROOT.name)
print("GIT_BRANCH:", GIT_BRANCH)
print("GIT_COMMIT:", GIT_COMMIT)
print("Working tree clean:", GIT_STATUS_PORCELAIN == "")
print("torch:", torch.__version__)


REPO_ROOT (relative label only): post-e50-baseline-inventory-cce171
GIT_BRANCH: research/post-e50-level-localization
GIT_COMMIT: 41ab6332f4ddcc110845763c1e12b31005f1e88a
Working tree clean: False
torch: 2.13.0+cpu


## 1. Modelo sagital congelado — `sagittal_spider` (read-only)

Se carga el checkpoint directamente desde `models/final/` (sin depender de variables de entorno
productivas), se calcula su SHA-256 real, y se compara contra el manifest. **No se reentrena, no
se modifica el checkpoint.** Se reutilizan (import read-only, sin modificar) las funciones de
arquitectura y preprocesamiento ya existentes en `ai_service/pfi_ai_service/` para garantizar
fidelidad exacta con el runtime productivo.


In [2]:
from ai_service.pfi_ai_service.model_architectures import build_checkpoint_model
from ai_service.pfi_ai_service.real_inference_runtime import (
    resize_image, robust_percentile_normalize, upsample_labels,
    connected_instances, class_name, is_background_class, LUMBAR_DISC_LEVELS,
)
from ai_service.pfi_ai_service.settings import MODEL_REGISTRY

MODEL_ID = "sagittal_spider"
CHECKPOINT_PATH = REPO_ROOT / "models" / "final" / "sagittal_spider_multiclass_final_best.pt"
MODEL_CARD_PATH = REPO_ROOT / "models" / "final" / "sagittal_spider_multiclass_final_best.pt.modelcard.md"
MANIFEST_PATH = REPO_ROOT / "models" / "final" / "sagittal_spider_multiclass_final_best.pt.manifest.json"

MODEL_AVAILABLE = CHECKPOINT_PATH.is_file()
GATE_A_checkpoint_identity = "FAIL"
sagittal_model = None
sagittal_runtime_meta = None
checkpoint_sha256 = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INFERENCE_DTYPE = "float32"

if MODEL_AVAILABLE:
    checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.is_file() else {}
    expected_sha = manifest.get("sha256")
    hash_match = (checkpoint_sha256 == expected_sha) if expected_sha else None

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    sagittal_model, sagittal_runtime_meta = build_checkpoint_model(MODEL_ID, checkpoint)
    sagittal_model.to(DEVICE)
    sagittal_model.eval()

    GATE_A_checkpoint_identity = "PASS" if (hash_match is not False) else "FAIL"
    if hash_match is False:
        warnings.append(f"Checkpoint SHA-256 mismatch vs manifest: actual={checkpoint_sha256} expected={expected_sha}")
else:
    warnings.append("sagittal_spider checkpoint not found on disk; cannot proceed with real inference.")

model_environment = {
    "model_id": MODEL_ID,
    "checkpoint_path": str(CHECKPOINT_PATH.relative_to(REPO_ROOT)),
    "checkpoint_sha256": checkpoint_sha256,
    "model_card_path": str(MODEL_CARD_PATH.relative_to(REPO_ROOT)) if MODEL_CARD_PATH.is_file() else None,
    "model_commit_source": manifest.get("sourceRepositoryCommit") if MODEL_AVAILABLE else None,
    "training_notebook": manifest.get("sourceTrainingNotebook") if MODEL_AVAILABLE else None,
    "device": str(DEVICE),
    "inference_dtype": INFERENCE_DTYPE,
    "target_size": list(sagittal_runtime_meta["targetSize"]) if sagittal_runtime_meta else None,
    "num_classes": sagittal_runtime_meta["numClasses"] if sagittal_runtime_meta else None,
    "class_names": MODEL_REGISTRY.get(MODEL_ID, {}).get("class_names"),
}
print(json.dumps({k: v for k, v in model_environment.items() if k != "class_names"}, indent=2, default=str))
print("class_names:", model_environment["class_names"])
print("GATE_A_checkpoint_identity:", GATE_A_checkpoint_identity)
if GATE_A_checkpoint_identity != "PASS":
    warnings.append("GATE A FAILED or checkpoint unavailable -- STOP condition per brief; downstream cells will guard on MODEL_AVAILABLE.")


{
  "model_id": "sagittal_spider",
  "checkpoint_path": "models\\final\\sagittal_spider_multiclass_final_best.pt",
  "checkpoint_sha256": "cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944",
  "model_card_path": "models\\final\\sagittal_spider_multiclass_final_best.pt.modelcard.md",
  "model_commit_source": "6013e160f45c9263fd4ae50e864ceb37245323e2",
  "training_notebook": "notebooks/45_gcs_spider_final_training.ipynb",
  "device": "cpu",
  "inference_dtype": "float32",
  "target_size": [
    256,
    256
  ],
  "num_classes": 4
}
class_names: {0: 'background', 1: 'vertebra_group', 2: 'canal', 3: 'disc_group'}
GATE_A_checkpoint_identity: PASS


## 2. Disponibilidad de SPIDER (read-only, sin descargar)

Se busca localmente evidencia de que el dataset SPIDER (no un dataset distinto con nombre
parecido) está disponible, respetando `PFI_POST_E50_SPIDER_ROOT` si está seteada. **No se asume
ningún ID de clase**: si se encuentra, se inspeccionaría primero su estructura real antes de usar
cualquier mapping.


In [3]:
def discover_spider_root() -> tuple[Path | None, str]:
    env_value = os.environ.get("PFI_POST_E50_SPIDER_ROOT")
    if env_value:
        candidate = Path(env_value)
        if candidate.exists():
            return candidate, "env:PFI_POST_E50_SPIDER_ROOT"
        warnings.append("PFI_POST_E50_SPIDER_ROOT is set but does not exist on disk.")

    anchor = None
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if parent.name == "PFI_MVP_Juntos":
            anchor = parent
            break
    if anchor is None:
        return None, "not_found:no_PFI_MVP_Juntos_ancestor"

    # Look for a directory plausibly named SPIDER (not other similarly-themed lumbar datasets)
    # under known project-adjacent locations only -- no internet access, no assumption of ID.
    candidates = []
    for base in [anchor, anchor.parent]:
        if base.is_dir():
            for child in base.iterdir():
                if child.is_dir() and "spider" in child.name.lower():
                    candidates.append(child)
    if candidates:
        return candidates[0], f"sibling_search:{candidates[0].name}"
    return None, "not_found:no_spider_named_directory_in_project_siblings"


SPIDER_ROOT, SPIDER_DISCOVERY_METHOD = discover_spider_root()
SPIDER_AVAILABLE = SPIDER_ROOT is not None

print("SPIDER available locally:", SPIDER_AVAILABLE)
print("Discovery method:", SPIDER_DISCOVERY_METHOD)
if not SPIDER_AVAILABLE:
    warnings.append("SPIDER dataset not found locally; held-out validation will be UNAVAILABLE (GATE G).")
    print("Held-out validation will be marked UNAVAILABLE. Real-DICOM exploratory path proceeds regardless.")


SPIDER available locally: False
Discovery method: not_found:no_spider_named_directory_in_project_siblings
Held-out validation will be marked UNAVAILABLE. Real-DICOM exploratory path proceeds regardless.


## 3. Fuente DICOM real (read-only, misma metodología que Notebooks 66/66B)


In [4]:
def _redact_relative(path: Path, anchor_name: str = "PFI_MVP_Juntos") -> str:
    parts = path.resolve().parts
    masked_name = f"<{opaque_id(path.name)}>{path.suffix}"
    if anchor_name in parts:
        idx = parts.index(anchor_name)
        dir_parts = list(parts[idx:-1])
        return "/".join([*dir_parts, masked_name])
    return masked_name


def discover_dicom_source() -> tuple[Path | None, str]:
    env_value = os.environ.get("PFI_POST_E50_DICOM_STUDY")
    if env_value:
        candidate = Path(env_value)
        if candidate.exists():
            return candidate, "env:PFI_POST_E50_DICOM_STUDY"
        warnings.append("PFI_POST_E50_DICOM_STUDY is set but does not exist on disk.")

    anchor = None
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if parent.name == "PFI_MVP_Juntos":
            anchor = parent
            break
    if anchor is None:
        return None, "not_found:no_PFI_MVP_Juntos_ancestor"

    sibling_names = ["PFI_RM_Lumbar_Final", "PFI_RM_Lumbar_Profesores"]
    candidates = []
    for name in sibling_names:
        sibling = anchor / name
        test_data = sibling / "test-data"
        if test_data.is_dir():
            candidates.extend(sorted(test_data.glob("*.zip")))
    if not candidates:
        return None, "not_found:no_test_data_zip_in_siblings"

    expected_sha = "1C058033AADAF9C72AF8F1B5D85DBBBDCB7706FDDA50B74537CE71A55B2227B1".lower()
    for cand in candidates:
        if sha256_file(cand) == expected_sha:
            return cand, f"sibling_search:{_redact_relative(cand)}"
    return candidates[0], f"sibling_search_no_hash_match:{_redact_relative(candidates[0])}"


DICOM_SOURCE_PATH, DICOM_SOURCE_METHOD = discover_dicom_source()
REAL_DICOM_USED = DICOM_SOURCE_PATH is not None

EXPECTED_STUDY_ZIP_SHA256 = "1C058033AADAF9C72AF8F1B5D85DBBBDCB7706FDDA50B74537CE71A55B2227B1".lower()
dicom_zip_sha256 = sha256_file(DICOM_SOURCE_PATH) if (REAL_DICOM_USED and DICOM_SOURCE_PATH.is_file()) else None

print("DICOM source found:", REAL_DICOM_USED)
print("Discovery method:", DICOM_SOURCE_METHOD)
print("SHA-256:", dicom_zip_sha256)
if dicom_zip_sha256 and dicom_zip_sha256 != EXPECTED_STUDY_ZIP_SHA256:
    warnings.append("Located DICOM zip SHA-256 does not match the historically expected value; treated as a different dataset, not corruption.")
elif not REAL_DICOM_USED:
    warnings.append("No real DICOM study located; real-DICOM exploratory localization will be BLOCKED.")


DICOM source found: True
Discovery method: sibling_search:PFI_MVP_Juntos/PFI_RM_Lumbar_Final/test-data/<c40f992f99ed>.zip
SHA-256: 1c058033aadaf9c72af8f1b5d85dbbbdcb7706fdda50b74537ce71a55b2227b1


## 4. Lectura DICOM en memoria y geometría (misma convención verificada, Sagittal T2 y T1)

Nunca se escribe un `.dcm` ni un array de píxeles dentro del repositorio.


In [5]:
@dataclass
class DicomInstance:
    series_folder: str
    entry_name: str
    dataset: "pydicom.dataset.FileDataset"


def _iter_dicom_entries(source: Path):
    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            for name in zf.namelist():
                if name.endswith(".dcm"):
                    with zf.open(name) as handle:
                        data = handle.read()
                    parts = name.split("/")
                    series_folder = parts[1] if len(parts) > 1 else "unknown"
                    yield series_folder, name, data
    elif source.is_dir():
        for path in source.rglob("*.dcm"):
            yield path.parent.name, str(path.relative_to(source)), path.read_bytes()


def load_instances_metadata_only(source: Path) -> list[DicomInstance]:
    out = []
    for series_folder, entry_name, data in _iter_dicom_entries(source):
        ds = pydicom.dcmread(io.BytesIO(data), stop_before_pixels=True)
        out.append(DicomInstance(series_folder, entry_name, ds))
    return out


def load_pixel_array_for_entry(source: Path, entry_name: str) -> np.ndarray:
    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source) as zf:
            with zf.open(entry_name) as handle:
                data = handle.read()
        ds = pydicom.dcmread(io.BytesIO(data))
    else:
        ds = pydicom.dcmread(source / entry_name)
    return ds.pixel_array.astype(np.float32)


def row_column_cosines(iop) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(iop, dtype=np.float64)
    return arr[:3], arr[3:]


def plane_normal(row_cos: np.ndarray, col_cos: np.ndarray) -> np.ndarray:
    return np.cross(row_cos, col_cos)


def pixel_to_patient_xyz(image_position, image_orientation, pixel_spacing, row, col) -> np.ndarray:
    ipp = np.asarray(image_position, dtype=np.float64)
    row_cos, col_cos = row_column_cosines(image_orientation)
    d_row, d_col = float(pixel_spacing[0]), float(pixel_spacing[1])
    return ipp + col * d_col * row_cos + row * d_row * col_cos


def sanitize_series_description(value) -> str:
    return str(value)[:64] if value is not None else ""


def classify_series_role(sample_ds, description_sanitized: str) -> str:
    if not hasattr(sample_ds, "ImageOrientationPatient") or sample_ds.ImageOrientationPatient is None:
        return "unknown"
    row_cos, col_cos = row_column_cosines(sample_ds.ImageOrientationPatient)
    normal = plane_normal(row_cos, col_cos)
    dominant_axis = int(np.argmax(np.abs(normal)))
    geometric_family = "sagittal" if dominant_axis == 0 else ("axial" if dominant_axis == 2 else "other")
    desc_lower = description_sanitized.lower()
    weighting = "t1" if "t1" in desc_lower else ("t2" if "t2" in desc_lower else None)
    if geometric_family == "sagittal" and weighting == "t1":
        return "sagittal_t1"
    if geometric_family == "sagittal" and weighting == "t2":
        return "sagittal_t2"
    if geometric_family == "axial" and weighting == "t2":
        return "axial_t2"
    return "other"


series_geometry: dict[str, dict] = {}
series_rows = []
STUDY_OPAQUE_ID = "unknown_study"
sag_t2_ids, sag_t1_ids = [], []

if REAL_DICOM_USED:
    instances = load_instances_metadata_only(DICOM_SOURCE_PATH)
    study_uid_raw = None
    by_series: dict[str, list[DicomInstance]] = {}
    for inst in instances:
        sid = str(inst.dataset.SeriesInstanceUID)
        by_series.setdefault(sid, []).append(inst)
        if study_uid_raw is None and hasattr(inst.dataset, "StudyInstanceUID"):
            study_uid_raw = str(inst.dataset.StudyInstanceUID)
    STUDY_OPAQUE_ID = opaque_id(study_uid_raw) if study_uid_raw else "unknown_study"

    for series_uid, insts in by_series.items():
        series_opaque = opaque_id(series_uid)
        sample = insts[0].dataset
        role = classify_series_role(sample, sanitize_series_description(getattr(sample, "SeriesDescription", None)))
        series_geometry[series_opaque] = {"instances": insts, "sample_ds": sample, "role": role}
        series_rows.append({"series_opaque_id": series_opaque, "candidate_role": role, "slice_count": len(insts)})

    series_inventory = pd.DataFrame(series_rows)
    sag_t2_ids = series_inventory.loc[series_inventory["candidate_role"] == "sagittal_t2", "series_opaque_id"].tolist()
    sag_t1_ids = series_inventory.loc[series_inventory["candidate_role"] == "sagittal_t1", "series_opaque_id"].tolist()
else:
    instances = []
    series_inventory = pd.DataFrame()

print("STUDY_OPAQUE_ID:", STUDY_OPAQUE_ID)
print("sagittal_t2 series:", sag_t2_ids)
print("sagittal_t1 series (secondary evidence only):", sag_t1_ids)


STUDY_OPAQUE_ID: 206aa67ee4e6
sagittal_t2 series: ['25aa08338722']
sagittal_t1 series (secondary evidence only): ['28ea4937243f']


In [6]:
def order_series_physically(series_opaque_id: str) -> list[tuple[float, "DicomInstance"]]:
    g = series_geometry[series_opaque_id]
    sample = g["sample_ds"]
    row_cos, col_cos = row_column_cosines(sample.ImageOrientationPatient)
    normal = plane_normal(row_cos, col_cos)
    scored = []
    for inst in g["instances"]:
        ipp = np.asarray(inst.dataset.ImagePositionPatient, dtype=np.float64)
        scored.append((float(np.dot(ipp, normal)), inst))
    scored.sort(key=lambda item: item[0])
    return scored


slice_ordering: dict[str, list] = {}
if REAL_DICOM_USED and sag_t2_ids:
    for sid in sag_t2_ids + sag_t1_ids:
        slice_ordering[sid] = order_series_physically(sid)
    print(f"Sagittal T2 physical slice count: {len(slice_ordering.get(sag_t2_ids[0], []))}" if sag_t2_ids else "No sagittal_t2 series.")
else:
    warnings.append("No sagittal_t2 series found in the real DICOM study; localization will be BLOCKED.")


Sagittal T2 physical slice count: 12


## 5. Inferencia sagital sobre TODA la serie (no un slice arbitrario)

Se ejecuta el modelo sobre cada slice físico de Sagittal T2 usando exactamente el preprocesamiento
productivo (`resize_image`: percentile-normalize 1-99 → uint8 → resize bilinear a `targetSize` →
`[0,1]`). Para cada slice se registran áreas por clase, número de componentes conexos de
`disc_group`, fracción de foreground y señales de calidad -- **sin elegir todavía** el "mejor"
slice de forma arbitraria.


In [7]:
GATE_B_inference_executable = "FAIL"
sagittal_slice_inventory_rows = []
slice_predictions: dict[int, dict] = {}  # slice_index -> {"prediction": arr, "confidence": arr, "ds": dataset}

if MODEL_AVAILABLE and REAL_DICOM_USED and sag_t2_ids:
    ref_sid = sag_t2_ids[0]
    ordered = slice_ordering[ref_sid]
    target_size = tuple(sagittal_runtime_meta["targetSize"])
    class_names_by_id = MODEL_REGISTRY.get(MODEL_ID, {}).get("class_names", {})
    disc_class_id = next((cid for cid, name in class_names_by_id.items() if name == "disc_group"), None)
    vertebra_class_id = next((cid for cid, name in class_names_by_id.items() if name == "vertebra_group"), None)
    canal_class_id = next((cid for cid, name in class_names_by_id.items() if name == "canal"), None)

    try:
        for slice_index, (scalar, inst) in enumerate(ordered):
            native = load_pixel_array_for_entry(DICOM_SOURCE_PATH, inst.entry_name)
            prepared = resize_image(native, target_size)
            tensor = torch.from_numpy(prepared[None, None]).float().to(DEVICE)
            with torch.inference_mode():
                logits = sagittal_model(tensor)
                probabilities = torch.softmax(logits, dim=1)[0]
            prediction = torch.argmax(probabilities, dim=0).cpu().numpy().astype(np.uint8)
            confidence = torch.max(probabilities, dim=0).values.cpu().numpy().astype(np.float32)
            slice_predictions[slice_index] = {"prediction": prediction, "confidence": confidence, "ds": inst.dataset, "physical_scalar": scalar}

            total_px = int(prediction.size)
            vertebra_area = int((prediction == vertebra_class_id).sum()) if vertebra_class_id is not None else 0
            disc_area = int((prediction == disc_class_id).sum()) if disc_class_id is not None else 0
            canal_area = int((prediction == canal_class_id).sum()) if canal_class_id is not None else 0
            foreground_fraction = float((prediction != 0).sum()) / total_px

            disc_components = connected_instances(prediction == disc_class_id) if disc_class_id is not None else []
            disc_component_count = len(disc_components)

            # Border clipping proxy: any foreground touching the 1px image border.
            border_touch = bool(
                prediction[0, :].any() or prediction[-1, :].any() or prediction[:, 0].any() or prediction[:, -1].any()
            )

            row_warn = []
            if disc_component_count == 0:
                row_warn.append("no_disc_components_detected")
            if foreground_fraction < 0.02:
                row_warn.append("very_low_foreground_fraction")
            if border_touch:
                row_warn.append("segmentation_touches_image_border")

            sagittal_slice_inventory_rows.append({
                "study_opaque_id": STUDY_OPAQUE_ID,
                "slice_index": slice_index,
                "physical_position": scalar,
                "vertebra_area": vertebra_area,
                "disc_area": disc_area,
                "canal_area": canal_area,
                "disc_component_count": disc_component_count,
                "foreground_fraction": foreground_fraction,
                "mean_confidence": float(confidence.mean()),
                "border_touch": border_touch,
                "candidate_quality": None,  # filled in Section 6
                "warnings": "; ".join(row_warn),
            })
        GATE_B_inference_executable = "PASS"
    except Exception as exc:
        warnings.append(f"Sagittal inference failed: {exc}")
        GATE_B_inference_executable = "FAIL"
else:
    warnings.append("Sagittal inference skipped: model or real DICOM sagittal_t2 unavailable.")

sagittal_slice_inventory = pd.DataFrame(sagittal_slice_inventory_rows)
print("GATE_B_inference_executable:", GATE_B_inference_executable)
sagittal_slice_inventory


GATE_B_inference_executable: PASS


,study_opaque_id,slice_index,physical_position,vertebra_area,disc_area,canal_area,disc_component_count,foreground_fraction,mean_confidence,border_touch,candidate_quality,warnings
0,206aa67ee4e6,0,-22.710411,108,10,0,0,0.001801,0.998623,False,None,no_disc_components_detected; very_low_foregrou...
1,206aa67ee4e6,1,-17.210416,1321,304,0,4,0.024796,0.996178,False,None,
2,206aa67ee4e6,2,-11.710441,4614,671,0,7,0.080643,0.993811,True,None,segmentation_touches_image_border
3,206aa67ee4e6,3,-6.210466,4467,933,215,7,0.085678,0.992438,True,None,segmentation_touches_image_border
4,206aa67ee4e6,4,-0.710491,4275,1053,1513,7,0.104385,0.993924,True,None,segmentation_touches_image_border
5,206aa67ee4e6,5,4.789488,5816,1081,2126,7,0.137680,0.991492,True,None,segmentation_touches_image_border
6,206aa67ee4e6,6,10.289413,4171,965,1556,7,0.102112,0.993288,True,None,segmentation_touches_image_border
7,206aa67ee4e6,7,15.789438,4382,727,9,7,0.078094,0.993875,True,None,segmentation_touches_image_border
8,206aa67ee4e6,8,21.289463,2633,412,0,5,0.046463,0.993452,False,None,
9,206aa67ee4e6,9,26.789388,406,30,0,1,0.006653,0.997890,False,None,very_low_foreground_fraction


## 6. `sagittal_slice_quality_score` — score determinístico (NO probabilidad)

No se asume que el índice medio del array es el mejor slice. Fórmula exacta, basada solo en
señales observables, normalizada por slice a `[0, 1]` y luego recortada:

```
sagittal_slice_quality_score =
    0.35 * canal_score          (canal_area / max(canal_area) entre slices de esta serie)
  + 0.25 * vertebra_score       (vertebra_area / max(vertebra_area) entre slices)
  + 0.25 * disc_score           (disc_area / max(disc_area) entre slices)
  + 0.15 * component_plausibility_score  (1.0 si 3<=disc_component_count<=6, decae fuera de ese rango)
  - 0.20 (penalización) si border_touch
  - 0.15 (penalización) si disc_component_count == 0
```

Resultado recortado a `[0, 1]`. No se llama "probability" en ningún artefacto.


In [8]:
def component_plausibility(n: int) -> float:
    if n == 0:
        return 0.0
    if 3 <= n <= 6:
        return 1.0
    distance = min(abs(n - 3), abs(n - 6))
    return max(0.0, 1.0 - 0.2 * distance)


if len(sagittal_slice_inventory):
    max_canal = max(sagittal_slice_inventory["canal_area"].max(), 1)
    max_vertebra = max(sagittal_slice_inventory["vertebra_area"].max(), 1)
    max_disc = max(sagittal_slice_inventory["disc_area"].max(), 1)

    scores = []
    for _, row in sagittal_slice_inventory.iterrows():
        canal_score = row["canal_area"] / max_canal
        vertebra_score = row["vertebra_area"] / max_vertebra
        disc_score = row["disc_area"] / max_disc
        component_score = component_plausibility(int(row["disc_component_count"]))
        score = 0.35 * canal_score + 0.25 * vertebra_score + 0.25 * disc_score + 0.15 * component_score
        if row["border_touch"]:
            score -= 0.20
        if row["disc_component_count"] == 0:
            score -= 0.15
        scores.append(float(np.clip(score, 0.0, 1.0)))

    sagittal_slice_inventory["candidate_quality"] = scores
    BEST_SLICE_INDEX = int(sagittal_slice_inventory.loc[sagittal_slice_inventory["candidate_quality"].idxmax(), "slice_index"])
    TOP_K = 5
    top_k_slices = sagittal_slice_inventory.sort_values("candidate_quality", ascending=False).head(TOP_K)["slice_index"].tolist()
    print("best slice:", BEST_SLICE_INDEX)
    print("top-K useful slices:", top_k_slices)
else:
    BEST_SLICE_INDEX = None
    top_k_slices = []
    warnings.append("No sagittal slice inventory available; cannot select best/useful slices.")

sagittal_slice_inventory


best slice: 5
top-K useful slices: [5, 4, 6, 3, 8]


,study_opaque_id,slice_index,physical_position,vertebra_area,disc_area,canal_area,disc_component_count,foreground_fraction,mean_confidence,border_touch,candidate_quality,warnings
0,206aa67ee4e6,0,-22.710411,108,10,0,0,0.001801,0.998623,False,0.000000,no_disc_components_detected; very_low_foregrou...
1,206aa67ee4e6,1,-17.210416,1321,304,0,4,0.024796,0.996178,False,0.277088,
2,206aa67ee4e6,2,-11.710441,4614,671,0,7,0.080643,0.993811,True,0.273513,segmentation_touches_image_border
3,206aa67ee4e6,3,-6.210466,4467,933,215,7,0.085678,0.992438,True,0.363181,segmentation_touches_image_border
4,206aa67ee4e6,4,-0.710491,4275,1053,1513,7,0.104385,0.993924,True,0.596368,segmentation_touches_image_border
5,206aa67ee4e6,5,4.789488,5816,1081,2126,7,0.137680,0.991492,True,0.770000,segmentation_touches_image_border
6,206aa67ee4e6,6,10.289413,4171,965,1556,7,0.102112,0.993288,True,0.578625,segmentation_touches_image_border
7,206aa67ee4e6,7,15.789438,4382,727,9,7,0.078094,0.993875,True,0.277973,segmentation_touches_image_border
8,206aa67ee4e6,8,21.289463,2633,412,0,5,0.046463,0.993452,False,0.358461,
9,206aa67ee4e6,9,26.789388,406,30,0,1,0.006653,0.997890,False,0.114390,very_low_foreground_fraction


## 7. Extracción de instancias discales por slice (`disc_group` → componentes conexos)

Se extraen componentes conexos de `disc_group` en cada uno de los `top_k_slices` (no solo el
mejor). **No se asume** que cada componente conexo es un disco válido: se aplican filtros de
ingeniería transparentes (tamaño mínimo, relación de aspecto, no-contacto con el borde salvo
justificación) documentados explícitamente, no umbrales clínicos.


In [9]:
MIN_COMPONENT_AREA_PX = 30
MAX_ASPECT_RATIO = 6.0


def component_geometry(mask: np.ndarray) -> dict:
    rows, cols = np.where(mask)
    area = int(mask.sum())
    centroid_row, centroid_col = float(rows.mean()), float(cols.mean())
    bbox = [int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1]
    height = bbox[1] - bbox[0]
    width = bbox[3] - bbox[2]
    major = max(height, width)
    minor = max(1, min(height, width))
    touches_border = bool(rows.min() == 0 or cols.min() == 0 or rows.max() == mask.shape[0] - 1 or cols.max() == mask.shape[1] - 1)
    return {
        "area_px": area, "centroid_row": centroid_row, "centroid_col": centroid_col,
        "bbox": bbox, "major_axis": major, "minor_axis": minor,
        "aspect_ratio": major / minor, "border_touch": touches_border,
    }


disc_instance_candidate_rows = []
disc_component_id_counter = 0

if MODEL_AVAILABLE and REAL_DICOM_USED and top_k_slices:
    target_size = tuple(sagittal_runtime_meta["targetSize"])
    for slice_index in top_k_slices:
        payload = slice_predictions[slice_index]
        prediction = payload["prediction"]
        ds = payload["ds"]
        components = connected_instances(prediction == disc_class_id) if disc_class_id is not None else []
        for component_mask in components:
            geom = component_geometry(component_mask)
            quality_flags = []
            if geom["area_px"] < MIN_COMPONENT_AREA_PX:
                quality_flags.append("below_min_area")
            if geom["aspect_ratio"] > MAX_ASPECT_RATIO:
                quality_flags.append("aspect_ratio_implausible")
            if geom["border_touch"]:
                quality_flags.append("touches_border")

            # Map centroid (in prediction grid) -> patient XYZ using this slice's own native
            # geometry, scaling from the resized grid back to native pixel indices.
            native_rows, native_cols = ds.Rows, ds.Columns
            row_native = geom["centroid_row"] * (int(native_rows) / target_size[0])
            col_native = geom["centroid_col"] * (int(native_cols) / target_size[1])
            centroid_xyz = pixel_to_patient_xyz(ds.ImagePositionPatient, ds.ImageOrientationPatient, ds.PixelSpacing, row_native, col_native)

            disc_component_id_counter += 1
            disc_instance_candidate_rows.append({
                "component_id": f"comp_{disc_component_id_counter:04d}",
                "slice_source": slice_index,
                "area_px": geom["area_px"],
                "centroid_row_col": [geom["centroid_row"], geom["centroid_col"]],
                "centroid_patient_xyz": centroid_xyz.tolist(),
                "bbox": geom["bbox"],
                "major_axis": geom["major_axis"],
                "minor_axis": geom["minor_axis"],
                "orientation_aspect_ratio": geom["aspect_ratio"],
                "border_touch": geom["border_touch"],
                "quality_flags": "; ".join(quality_flags),
                "accepted": len(quality_flags) == 0 or (quality_flags == ["touches_border"]),
            })

disc_instance_candidates = pd.DataFrame(disc_instance_candidate_rows)
GATE_C_disc_instance_extraction = "FAIL"
if len(disc_instance_candidates):
    accepted = disc_instance_candidates[disc_instance_candidates["accepted"]]
    GATE_C_disc_instance_extraction = "PASS" if len(accepted) > 0 else "PARTIAL"
print("GATE_C_disc_instance_extraction:", GATE_C_disc_instance_extraction)
disc_instance_candidates


GATE_C_disc_instance_extraction: PASS


,component_id,slice_source,area_px,centroid_row_col,centroid_patient_xyz,bbox,major_axis,minor_axis,orientation_aspect_ratio,border_touch,quality_flags,accepted
0,comp_0001,5,76,"[5.934210526315789, 119.57894736842105]","[-5.0889792748684215, 41.26356975750001, 177.1...","[4, 9, 109, 131]",22,5,4.400000,False,,True
1,comp_0002,5,120,"[33.28333333333333, 115.41666666666667]","[-4.908563560699999, 36.349041534, 145.1167809...","[30, 37, 102, 129]",27,7,3.857143,False,,True
2,comp_0003,5,120,"[61.575, 111.54166666666667]","[-4.696724091949999, 31.76923700400001, 111.97...","[58, 65, 97, 127]",30,7,4.285714,False,,True
3,comp_0004,5,160,"[90.73125, 107.35]","[-4.4900122979375, 26.817458601187496, 77.8118...","[87, 95, 92, 123]",31,8,3.875000,False,,True
4,comp_0005,5,189,"[120.85714285714286, 100.64021164021165]","[-4.4156279199999995, 18.917099710000002, 42.5...","[117, 125, 85, 117]",32,8,4.000000,False,,True
5,comp_0006,5,207,"[152.57004830917873, 96.69565217391305]","[-4.154819879130434, 14.250475124347824, 5.364...","[148, 158, 83, 113]",30,10,3.000000,False,,True
6,comp_0007,5,209,"[184.19617224880383, 100.85645933014354]","[-3.421033646287082, 19.069984093492817, -31.6...","[176, 195, 89, 116]",27,19,1.421053,False,,True
7,comp_0008,4,76,"[6.105263157894737, 120.23684210526316]","[0.44483233673684164, 41.758756565526326, 177....","[4, 9, 109, 133]",24,5,4.800000,False,,True
8,comp_0009,4,114,"[33.14912280701754, 115.6842105263158]","[0.5976723693157879, 36.38786985552632, 145.34...","[30, 37, 103, 130]",27,7,3.857143,False,,True
9,comp_0010,4,122,"[61.69672131147541, 112.25409836065573]","[0.8395142263770486, 32.32832837409837, 111.90...","[59, 65, 97, 128]",31,6,5.166667,False,,True


## 8. Consenso multi-slice — `DiscInstanceCandidate`

Se agrupan componentes aceptados de distintos slices que corresponden al **mismo disco físico**
usando proximidad de centroide en patient-space (criterio geométrico, no de aprendizaje). Umbral
de ingeniería: `CONSENSUS_DISTANCE_THRESHOLD_MM = 15.0` (aprox. 2-3 slice-spacings de Notebook 66),
documentado explícitamente, no clínico.


In [10]:
CONSENSUS_DISTANCE_THRESHOLD_MM = 15.0

disc_instances_consensus_rows = []
if len(disc_instance_candidates):
    accepted = disc_instance_candidates[disc_instance_candidates["accepted"]].copy()
    accepted["centroid_xyz_arr"] = accepted["centroid_patient_xyz"].apply(np.array)

    # Union-find by centroid proximity across (possibly) different slices.
    n = len(accepted)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[ry] = rx

    rows_list = accepted.reset_index(drop=True)
    for i in range(n):
        for j in range(i + 1, n):
            dist = float(np.linalg.norm(rows_list.loc[i, "centroid_xyz_arr"] - rows_list.loc[j, "centroid_xyz_arr"]))
            if dist <= CONSENSUS_DISTANCE_THRESHOLD_MM:
                union(i, j)

    groups: dict[int, list[int]] = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)

    for instance_id, (_, indices) in enumerate(sorted(groups.items(), key=lambda kv: kv[0])):
        members = rows_list.loc[indices]
        centroids = np.stack(members["centroid_xyz_arr"].to_list())
        mean_centroid = centroids.mean(axis=0)
        spread_mm = float(np.max(np.linalg.norm(centroids - mean_centroid, axis=1))) if len(members) > 1 else 0.0
        representative = members.loc[members["area_px"].idxmax()]
        disc_instances_consensus_rows.append({
            "instance_id": f"disc_inst_{instance_id:02d}",
            "supporting_slices": sorted(members["slice_source"].tolist()),
            "supporting_slice_count": len(members),
            "representative_slice": int(representative["slice_source"]),
            "centroid_patient_xyz": mean_centroid.tolist(),
            "centroid_spread_mm": spread_mm,
            "area_mean_px": float(members["area_px"].mean()),
            "area_std_px": float(members["area_px"].std()) if len(members) > 1 else 0.0,
        })

disc_instances_consensus = pd.DataFrame(disc_instances_consensus_rows)

GATE_D_multi_slice_consensus = "FAIL"
if len(disc_instances_consensus):
    multi_support = (disc_instances_consensus["supporting_slice_count"] > 1).sum()
    GATE_D_multi_slice_consensus = "PASS" if multi_support > 0 else "PARTIAL"
print("GATE_D_multi_slice_consensus:", GATE_D_multi_slice_consensus)
disc_instances_consensus


GATE_D_multi_slice_consensus: PASS


,instance_id,supporting_slices,supporting_slice_count,representative_slice,centroid_patient_xyz,centroid_spread_mm,area_mean_px,area_std_px
0,disc_inst_00,"[3, 4, 5, 6]",4,5,"[-2.259055810919259, 42.6881095392793, 176.781...",8.529525,70.25,6.751543
1,disc_inst_01,"[3, 4, 5, 6]",4,5,"[-2.1153143112739308, 37.149452398238026, 145....",8.430251,108.75,10.045729
2,disc_inst_02,"[3, 4, 5, 6, 8]",5,4,"[-5.737115985364591, 32.91765992849467, 111.69...",15.434232,102.00,30.724583
3,disc_inst_03,"[3, 4, 5, 6, 8]",5,4,"[-5.564682512292885, 27.29867155155144, 77.605...",15.421157,140.60,34.107184
4,disc_inst_04,"[3, 4, 5, 6, 8]",5,4,"[-5.512439425166436, 18.91452385163152, 42.159...",15.436761,169.00,27.504545
5,disc_inst_05,"[3, 4, 5, 6, 8]",5,5,"[-5.253839782797391, 14.246216147374302, 5.164...",15.404191,173.60,40.820338
6,disc_inst_06,"[3, 4, 5, 6, 8]",5,5,"[-4.491563245922584, 19.753092330370826, -31.4...",15.429970,160.40,60.677838


## 9. Eje cráneo-caudal en patient-space (`spine_axis_vector`)

**No** se ordena por fila de imagen ascendente. Se calcula un eje de elongación mediante PCA
sobre los píxeles de `vertebra_group` (o `canal` si vertebra no está disponible) del mejor slice,
convertidos a patient XYZ. El signo del eje se fija usando el eje Z real de paciente DICOM (LPS:
`Z+` = superior): se define `spine_axis_vector` apuntando en sentido **caudal** (hacia los pies),
de forma que `position_along_spine_mm` creciente = más caudal — sin depender de que la imagen
esté "derecha" en el array.


In [11]:
spine_axis_vector = None
GATE_E_anatomical_ordering = "FAIL"

if MODEL_AVAILABLE and REAL_DICOM_USED and BEST_SLICE_INDEX is not None:
    payload = slice_predictions[BEST_SLICE_INDEX]
    prediction, ds = payload["prediction"], payload["ds"]
    target_size = tuple(sagittal_runtime_meta["targetSize"])

    axis_class_id = vertebra_class_id if vertebra_class_id is not None and (prediction == vertebra_class_id).sum() > 0 else canal_class_id
    axis_mask = prediction == axis_class_id
    rows, cols = np.where(axis_mask)
    if rows.size >= 2:
        native_rows_n, native_cols_n = int(ds.Rows), int(ds.Columns)
        points_xyz = []
        # Subsample for speed if the mask is large.
        stride = max(1, rows.size // 2000)
        for r, c in zip(rows[::stride], cols[::stride]):
            r_native = r * (native_rows_n / target_size[0])
            c_native = c * (native_cols_n / target_size[1])
            points_xyz.append(pixel_to_patient_xyz(ds.ImagePositionPatient, ds.ImageOrientationPatient, ds.PixelSpacing, r_native, c_native))
        points_xyz = np.array(points_xyz)
        centered = points_xyz - points_xyz.mean(axis=0)
        _, _, vt = np.linalg.svd(centered, full_matrices=False)
        principal = vt[0]
        principal = principal / np.linalg.norm(principal)
        # Ensure the axis points caudally (patient Z decreasing = toward feet in LPS).
        if principal[2] > 0:
            principal = -principal
        spine_axis_vector = principal
        GATE_E_anatomical_ordering = "PASS"
        print("spine_axis_vector (points caudally):", spine_axis_vector)
    else:
        warnings.append("Not enough foreground pixels to derive spine axis via PCA.")
else:
    warnings.append("Spine axis could not be computed: model/DICOM/best-slice unavailable.")

if spine_axis_vector is not None and len(disc_instances_consensus):
    reference_origin = np.array(disc_instances_consensus.iloc[0]["centroid_patient_xyz"])
    positions = []
    for _, row in disc_instances_consensus.iterrows():
        centroid = np.array(row["centroid_patient_xyz"])
        positions.append(float(np.dot(centroid - reference_origin, spine_axis_vector)))
    disc_instances_consensus["position_along_spine_mm"] = positions
    disc_instances_consensus.sort_values("position_along_spine_mm", inplace=True)
    disc_instances_consensus.reset_index(drop=True, inplace=True)
    disc_instances_consensus["ordinal_cranial_to_caudal"] = range(1, len(disc_instances_consensus) + 1)
else:
    GATE_E_anatomical_ordering = "FAIL" if GATE_E_anatomical_ordering != "PASS" else "PARTIAL"

print("GATE_E_anatomical_ordering:", GATE_E_anatomical_ordering)
disc_instances_consensus


spine_axis_vector (points caudally):

 [ 0.0050791  -0.16094255 -0.98695071]
GATE_E_anatomical_ordering: PASS


,instance_id,supporting_slices,supporting_slice_count,representative_slice,centroid_patient_xyz,centroid_spread_mm,area_mean_px,area_std_px,position_along_spine_mm,ordinal_cranial_to_caudal
0,disc_inst_00,"[3, 4, 5, 6]",4,5,"[-2.259055810919259, 42.6881095392793, 176.781...",8.529525,70.25,6.751543,0.000000,1
1,disc_inst_01,"[3, 4, 5, 6]",4,5,"[-2.1153143112739308, 37.149452398238026, 145....",8.430251,108.75,10.045729,32.109374,2
2,disc_inst_02,"[3, 4, 5, 6, 8]",5,4,"[-5.737115985364591, 32.91765992849467, 111.69...",15.434232,102.00,30.724583,65.791183,3
3,disc_inst_03,"[3, 4, 5, 6, 8]",5,4,"[-5.564682512292885, 27.29867155155144, 77.605...",15.421157,140.60,34.107184,100.341028,4
4,disc_inst_04,"[3, 4, 5, 6, 8]",5,4,"[-5.512439425166436, 18.91452385163152, 42.159...",15.436761,169.00,27.504545,136.674793,5
5,disc_inst_05,"[3, 4, 5, 6, 8]",5,5,"[-5.253839782797391, 14.246216147374302, 5.164...",15.404191,173.60,40.820338,173.939593,6
6,disc_inst_06,"[3, 4, 5, 6, 8]",5,5,"[-4.491563245922584, 19.753092330370826, -31.4...",15.429970,160.40,60.677838,209.197200,7


## 10. Soporte de instancias vertebrales (investigado, no asumido)

Se investiga si `vertebra_group` produce componentes separables por vértebra individual en el
mejor slice. **No se asume que sí.** Si los componentes conexos de `vertebra_group` no se
separan de forma plausible (p. ej. un único blob grande fusionado), se usa la máscara solo como
contexto para el eje espinal (Sección 9), sin generar `vertebral instance candidates` ni inventar
nombres `L1`, `L2`, etc.


In [12]:
vertebral_instance_candidates_rows = []
VERTEBRA_INSTANCES_SEPARABLE = False

if MODEL_AVAILABLE and REAL_DICOM_USED and BEST_SLICE_INDEX is not None and vertebra_class_id is not None:
    payload = slice_predictions[BEST_SLICE_INDEX]
    prediction = payload["prediction"]
    vertebra_components = connected_instances(prediction == vertebra_class_id)
    # Heuristic plausibility check: a fused single blob is NOT separable; multiple similarly
    # sized components spread along the spine axis direction ARE plausible vertebral bodies.
    if len(vertebra_components) >= 3:
        areas = [int(c.sum()) for c in vertebra_components]
        area_cv = float(np.std(areas) / np.mean(areas)) if np.mean(areas) > 0 else 999.0
        VERTEBRA_INSTANCES_SEPARABLE = area_cv < 0.6  # engineering threshold: fused blobs are highly non-uniform
    if VERTEBRA_INSTANCES_SEPARABLE:
        for i, component in enumerate(vertebra_components):
            geom = component_geometry(component)
            vertebral_instance_candidates_rows.append({
                "vertebra_component_id": f"vert_{i:02d}",
                "slice_source": BEST_SLICE_INDEX,
                "area_px": geom["area_px"],
                "centroid_row_col": [geom["centroid_row"], geom["centroid_col"]],
            })

vertebral_instance_candidates = pd.DataFrame(vertebral_instance_candidates_rows)
print("VERTEBRA_INSTANCES_SEPARABLE:", VERTEBRA_INSTANCES_SEPARABLE)
if not VERTEBRA_INSTANCES_SEPARABLE:
    limitations.append(
        "vertebra_group did not produce a plausible set of separable per-vertebra instances in "
        "this run; the mask was used only as spine-axis context, not as a vertebral anchor. No "
        "vertebral instance names (L1, L2, ...) were fabricated."
    )
vertebral_instance_candidates


VERTEBRA_INSTANCES_SEPARABLE: True


,vertebra_component_id,slice_source,area_px,centroid_row_col
0,vert_00,5,97,"[1.7422680412371134, 122.97938144329896]"
1,vert_01,5,127,"[6.2362204724409445, 152.5275590551181]"
2,vert_02,5,538,"[19.659851301115243, 118.53159851301115]"
3,vert_03,5,227,"[31.378854625550662, 156.72246696035242]"
4,vert_04,5,600,"[47.89, 114.69166666666666]"
5,vert_05,5,306,"[57.627450980392155, 155.76797385620915]"
6,vert_06,5,653,"[75.82235834609494, 109.71056661562021]"
7,vert_07,5,336,"[87.36011904761905, 151.60416666666666]"
8,vert_08,5,662,"[105.57552870090635, 103.6737160120846]"
9,vert_09,5,375,"[118.84533333333333, 146.856]"


## 11. Naming absoluto de niveles — tres tareas separadas, con abstención

Se separan explícitamente **tres** tareas distintas (no dos):

1. **DISC INSTANCE DETECTION** (Secciones 7-8): cuántos discos hay y dónde están. El campo de
   visión sagital puede legítimamente contener discos fuera del rango lumbar objetivo
   (torácicos bajos, por ejemplo), así que `disc_instances_detected != 5` **no** es, por sí
   solo, evidencia de que la detección esté mal — esa conclusión no se asume aquí.
2. **TARGET LUMBAR LEVEL SELECTION**: de todas las instancias detectadas, ¿cuáles 5 (o menos)
   corresponden efectivamente a la ventana lumbar `L1-L2...L5-S1`? Esto requiere un anchor
   anatómico independiente de la simple cuenta de candidatos.
3. **ABSOLUTE LEVEL NAMING**: asignar la etiqueta exacta a cada instancia de la ventana ya
   seleccionada.

**Anchors disponibles en este modelo/estudio, evaluados honestamente:**

- (A) referencia SPIDER: no disponible en esta corrida (Sección 2).
- (B) estructura sacral/inferior demostrable: el checkpoint `sagittal_spider` **no** tiene una
  clase dedicada de sacro/S1 — no se puede demostrar con las máscaras existentes.
- (C) conteo de instancias vertebrales + anchor inferior validado: requiere (B), que no existe.
- (D) metadata de dataset/referencia: no aplica al DICOM real (no es SPIDER).

Sin (A)-(D) genuinamente disponibles, la selección de la ventana lumbar objetivo (tarea 2) queda
en estado **`target_lumbar_window_status = AMBIGUOUS`** -- no `RESOLVED` -- **independientemente
de cuántas instancias se hayan detectado**. No se asume `lowest_five = L1-L2...L5-S1` ni
`highest_five = L1-L2...L5-S1`: ambas son la misma clase de suposición no demostrada que este
notebook evita deliberadamente. En consecuencia, la tarea 3 (naming absoluto) también abstiene.
La causa primaria reportada es **`ABSOLUTE_ANATOMICAL_ANCHOR_UNAVAILABLE`**, no el conteo de
instancias -- el conteo se registra únicamente como información auxiliar en
`anchor_quality_gates`, nunca como causa suficiente.


In [13]:
anchor_quality_gates = {}
GATE_F_absolute_level_naming = "FAIL"
absolute_level_status = "abstain"
target_lumbar_window_status = "UNAVAILABLE"
abstain_primary_reason = None
levels_assigned = 0

if len(disc_instances_consensus):
    n_instances = len(disc_instances_consensus)

    # Auxiliary information only -- NEVER treated as sufficient proof of correct/incorrect
    # instance detection, and never used alone to resolve the target lumbar window.
    anchor_quality_gates["instance_count_is_five"] = n_instances == 5
    anchor_quality_gates["instance_count_in_plausible_range"] = 4 <= n_instances <= 6

    caudal_instance = disc_instances_consensus.iloc[-1]
    cranial_instance = disc_instances_consensus.iloc[0]
    caudal_slice_candidates = disc_instance_candidates[
        (disc_instance_candidates["slice_source"] == caudal_instance["representative_slice"])
    ]
    cranial_slice_candidates = disc_instance_candidates[
        (disc_instance_candidates["slice_source"] == cranial_instance["representative_slice"])
    ]
    anchor_quality_gates["no_caudal_clipping"] = bool((~caudal_slice_candidates["border_touch"]).any()) if len(caudal_slice_candidates) else False
    anchor_quality_gates["no_cranial_clipping"] = bool((~cranial_slice_candidates["border_touch"]).any()) if len(cranial_slice_candidates) else False
    anchor_quality_gates["multi_slice_support_majority"] = bool((disc_instances_consensus["supporting_slice_count"] > 1).mean() >= 0.5)

    # Genuine anatomical anchor sources (A)-(D) from the markdown above -- ALL unavailable in
    # this run. This, not the instance count, is what drives target-window resolution.
    anchor_sources_available = {
        "spider_reference_labels": SPIDER_AVAILABLE,
        "demonstrable_sacral_structure": False,  # sagittal_spider has no dedicated sacrum/S1 class
        "validated_vertebral_instance_anchor": False,  # depends on the sacral anchor above
        "dataset_reference_metadata": False,  # not applicable to a real, non-SPIDER DICOM study
    }
    if any(anchor_sources_available.values()):
        target_lumbar_window_status = "RESOLVED"
    else:
        target_lumbar_window_status = "AMBIGUOUS"

    if target_lumbar_window_status == "RESOLVED":
        # Extension point for a future run where an anchor source actually exists (e.g. Notebook
        # 67A with SPIDER). Not implemented here because no anchor is genuinely available.
        GATE_F_absolute_level_naming = "PARTIAL"
        abstain_primary_reason = "ANCHOR_RESOLVED_BUT_NAMING_NOT_IMPLEMENTED"
    else:
        disc_instances_consensus["level"] = None
        disc_instances_consensus["level_status"] = "abstain"
        absolute_level_status = "abstain"
        levels_assigned = 0
        abstain_primary_reason = "ABSOLUTE_ANATOMICAL_ANCHOR_UNAVAILABLE"
        GATE_F_absolute_level_naming = "PARTIAL"
        warnings.append(
            f"Absolute level naming ABSTAINED: {abstain_primary_reason} "
            f"(target_lumbar_window_status={target_lumbar_window_status}; "
            f"instance_count={n_instances} is auxiliary information only, not the cause)."
        )
else:
    target_lumbar_window_status = "UNAVAILABLE"
    abstain_primary_reason = "NO_DISC_INSTANCES_DETECTED"
    GATE_F_absolute_level_naming = "FAIL"
    warnings.append("No disc instances available; absolute level naming ABSTAINED (nothing to name).")

print("anchor_quality_gates:", anchor_quality_gates)
print("target_lumbar_window_status:", target_lumbar_window_status)
print("absolute_level_status:", absolute_level_status)
print("abstain_primary_reason:", abstain_primary_reason)
print("levels_assigned:", levels_assigned)
print("GATE_F_absolute_level_naming:", GATE_F_absolute_level_naming)
disc_instances_consensus


anchor_quality_gates: {'instance_count_is_five': False, 'instance_count_in_plausible_range': False, 'no_caudal_clipping': True, 'no_cranial_clipping': True, 'multi_slice_support_majority': True}
target_lumbar_window_status: AMBIGUOUS
absolute_level_status: abstain
abstain_primary_reason: ABSOLUTE_ANATOMICAL_ANCHOR_UNAVAILABLE
levels_assigned: 0
GATE_F_absolute_level_naming: PARTIAL


,instance_id,supporting_slices,supporting_slice_count,representative_slice,centroid_patient_xyz,centroid_spread_mm,area_mean_px,area_std_px,position_along_spine_mm,ordinal_cranial_to_caudal,level,level_status
0,disc_inst_00,"[3, 4, 5, 6]",4,5,"[-2.259055810919259, 42.6881095392793, 176.781...",8.529525,70.25,6.751543,0.000000,1,None,abstain
1,disc_inst_01,"[3, 4, 5, 6]",4,5,"[-2.1153143112739308, 37.149452398238026, 145....",8.430251,108.75,10.045729,32.109374,2,None,abstain
2,disc_inst_02,"[3, 4, 5, 6, 8]",5,4,"[-5.737115985364591, 32.91765992849467, 111.69...",15.434232,102.00,30.724583,65.791183,3,None,abstain
3,disc_inst_03,"[3, 4, 5, 6, 8]",5,4,"[-5.564682512292885, 27.29867155155144, 77.605...",15.421157,140.60,34.107184,100.341028,4,None,abstain
4,disc_inst_04,"[3, 4, 5, 6, 8]",5,4,"[-5.512439425166436, 18.91452385163152, 42.159...",15.436761,169.00,27.504545,136.674793,5,None,abstain
5,disc_inst_05,"[3, 4, 5, 6, 8]",5,5,"[-5.253839782797391, 14.246216147374302, 5.164...",15.404191,173.60,40.820338,173.939593,6,None,abstain
6,disc_inst_06,"[3, 4, 5, 6, 8]",5,5,"[-4.491563245922584, 19.753092330370826, -31.4...",15.429970,160.40,60.677838,209.197200,7,None,abstain


## 12. Confidence scores (determinísticos, NO probabilidades)

Se separan explícitamente dos scores:

```
instance_confidence =
    0.40 * multi_slice_support_score   (min(supporting_slice_count / 3, 1.0))
  + 0.30 * centroid_stability_score    (1.0 si centroid_spread_mm < 5mm, decae linealmente hasta 0 a 20mm)
  + 0.30 * segmentation_confidence_score (mean_confidence del slice representativo)

level_naming_confidence =
    0.30 * count_score        (1.0 si exactamente 5 instancias, si no según anchor_quality_gates)
  + 0.25 * coverage_score     (1.0 si no_caudal_clipping y no_cranial_clipping)
  + 0.25 * anchor_score       (1.0 si anchor_quality_gates confirma heuristic candidate, 0 si abstain)
  + 0.20 * spacing_consistency_score (1.0 si std(position_along_spine_mm diffs)/mean bajo, ver Notebook 66)
```

Ambos en `[0, 1]`, documentados aquí exactamente, nunca llamados "probability".


In [14]:
def compute_instance_confidence(row) -> float:
    multi_slice_score = min(row["supporting_slice_count"] / 3.0, 1.0)
    spread = row["centroid_spread_mm"]
    stability_score = float(np.clip(1.0 - spread / 20.0, 0.0, 1.0))
    payload = slice_predictions.get(row["representative_slice"])
    seg_conf = float(sagittal_slice_inventory.loc[sagittal_slice_inventory["slice_index"] == row["representative_slice"], "mean_confidence"].iloc[0]) if payload else 0.0
    return round(0.40 * multi_slice_score + 0.30 * stability_score + 0.30 * seg_conf, 4)


if len(disc_instances_consensus):
    disc_instances_consensus["instance_confidence"] = disc_instances_consensus.apply(compute_instance_confidence, axis=1)

    count_score = 1.0 if anchor_quality_gates.get("instance_count_is_five") else (0.5 if anchor_quality_gates.get("instance_count_in_plausible_range") else 0.0)
    coverage_score = 1.0 if (anchor_quality_gates.get("no_caudal_clipping") and anchor_quality_gates.get("no_cranial_clipping")) else 0.0
    anchor_score = 1.0 if absolute_level_status in ("inferred", "partial_inferred") else 0.0
    if len(disc_instances_consensus) > 1 and "position_along_spine_mm" in disc_instances_consensus.columns:
        diffs = np.diff(sorted(disc_instances_consensus["position_along_spine_mm"]))
        spacing_consistency_score = float(np.clip(1.0 - (np.std(diffs) / max(np.mean(diffs), 1e-6)), 0.0, 1.0)) if len(diffs) else 0.0
    else:
        spacing_consistency_score = 0.0

    LEVEL_NAMING_CONFIDENCE = round(0.30 * count_score + 0.25 * coverage_score + 0.25 * anchor_score + 0.20 * spacing_consistency_score, 4)
    print("level_naming_confidence:", LEVEL_NAMING_CONFIDENCE)
else:
    LEVEL_NAMING_CONFIDENCE = 0.0

disc_instances_consensus


level_naming_confidence: 0.4403


,instance_id,supporting_slices,supporting_slice_count,representative_slice,centroid_patient_xyz,centroid_spread_mm,area_mean_px,area_std_px,position_along_spine_mm,ordinal_cranial_to_caudal,level,level_status,instance_confidence
0,disc_inst_00,"[3, 4, 5, 6]",4,5,"[-2.259055810919259, 42.6881095392793, 176.781...",8.529525,70.25,6.751543,0.000000,1,None,abstain,0.8695
1,disc_inst_01,"[3, 4, 5, 6]",4,5,"[-2.1153143112739308, 37.149452398238026, 145....",8.430251,108.75,10.045729,32.109374,2,None,abstain,0.8710
2,disc_inst_02,"[3, 4, 5, 6, 8]",5,4,"[-5.737115985364591, 32.91765992849467, 111.69...",15.434232,102.00,30.724583,65.791183,3,None,abstain,0.7667
3,disc_inst_03,"[3, 4, 5, 6, 8]",5,4,"[-5.564682512292885, 27.29867155155144, 77.605...",15.421157,140.60,34.107184,100.341028,4,None,abstain,0.7669
4,disc_inst_04,"[3, 4, 5, 6, 8]",5,4,"[-5.512439425166436, 18.91452385163152, 42.159...",15.436761,169.00,27.504545,136.674793,5,None,abstain,0.7666
5,disc_inst_05,"[3, 4, 5, 6, 8]",5,5,"[-5.253839782797391, 14.246216147374302, 5.164...",15.404191,173.60,40.820338,173.939593,6,None,abstain,0.7664
6,disc_inst_06,"[3, 4, 5, 6, 8]",5,5,"[-4.491563245922584, 19.753092330370826, -31.4...",15.429970,160.40,60.677838,209.197200,7,None,abstain,0.7660


## 12b. `ordered_disc_instance_profile` — perfil cranial → caudal (sin asignar niveles)

Se perfila cada instancia detectada, ordenada cranial → caudal, **sin** eliminar candidatos y
**sin** asignar niveles -- es evidencia descriptiva para investigar la ventana lumbar objetivo en
un futuro notebook (67A), no una decisión.


In [15]:
def grid_pixel_spacing_mm(ds, target_size) -> tuple[float, float]:
    native_rows, native_cols = int(ds.Rows), int(ds.Columns)
    row_spacing = float(ds.PixelSpacing[0]) * (native_rows / target_size[0])
    col_spacing = float(ds.PixelSpacing[1]) * (native_cols / target_size[1])
    return row_spacing, col_spacing


ordered_disc_instance_profile_rows = []
if len(disc_instances_consensus) and "position_along_spine_mm" in disc_instances_consensus.columns:
    target_size = tuple(sagittal_runtime_meta["targetSize"])
    ordered_df = disc_instances_consensus.sort_values("ordinal_cranial_to_caudal").reset_index(drop=True)
    positions = ordered_df["position_along_spine_mm"].tolist()

    for i, row in ordered_df.iterrows():
        rep_slice = row["representative_slice"]
        rep_ds = slice_predictions[rep_slice]["ds"]
        row_spacing_mm, col_spacing_mm = grid_pixel_spacing_mm(rep_ds, target_size)
        pixel_area_mm2 = row_spacing_mm * col_spacing_mm

        rep_candidates = disc_instance_candidates[disc_instance_candidates["slice_source"] == rep_slice]
        supporting_candidates = disc_instance_candidates[disc_instance_candidates["slice_source"].isin(row["supporting_slices"])]
        median_area_mm2 = float(supporting_candidates["area_px"].median() * pixel_area_mm2) if len(supporting_candidates) else None
        bbox_size = None
        border_touch_any = False
        quality_flags_union = set()
        if len(rep_candidates):
            bbox = rep_candidates.iloc[0]["bbox"]
            bbox_size = [bbox[1] - bbox[0], bbox[3] - bbox[2]]
        if len(supporting_candidates):
            border_touch_any = bool(supporting_candidates["border_touch"].any())
            for flags in supporting_candidates["quality_flags"]:
                if flags:
                    quality_flags_union.update(flags.split("; "))

        ordered_disc_instance_profile_rows.append({
            "ordered_index": i + 1,
            "instance_id": row["instance_id"],
            "supporting_slice_count": int(row["supporting_slice_count"]),
            "representative_slice": int(rep_slice),
            "position_along_spine_mm": float(row["position_along_spine_mm"]),
            "distance_to_previous_mm": float(positions[i] - positions[i - 1]) if i > 0 else None,
            "distance_to_next_mm": float(positions[i + 1] - positions[i]) if i < len(positions) - 1 else None,
            "centroid_spread_mm": float(row["centroid_spread_mm"]),
            "median_area_mm2": median_area_mm2,
            "bbox_size": bbox_size,
            "border_touch": border_touch_any,
            "instance_confidence": float(row["instance_confidence"]),
            "quality_flags": "; ".join(sorted(quality_flags_union)),
        })

ordered_disc_instance_profile = pd.DataFrame(ordered_disc_instance_profile_rows)
ordered_disc_instance_profile


,ordered_index,instance_id,supporting_slice_count,representative_slice,position_along_spine_mm,distance_to_previous_mm,distance_to_next_mm,centroid_spread_mm,median_area_mm2,bbox_size,border_touch,instance_confidence,quality_flags
0,1,disc_inst_00,4,5,0.000000,NaN,32.109374,8.529525,211.459747,"[5, 22]",False,0.8695,
1,2,disc_inst_01,4,5,32.109374,32.109374,33.681809,8.430251,211.459747,"[5, 22]",False,0.8710,
2,3,disc_inst_02,5,4,65.791183,33.681809,34.549845,15.434232,168.893175,"[5, 24]",False,0.7667,
3,4,disc_inst_03,5,4,100.341028,34.549845,36.333764,15.421157,168.893175,"[5, 24]",False,0.7669,
4,5,disc_inst_04,5,4,136.674793,36.333764,37.264800,15.436761,168.893175,"[5, 24]",False,0.7666,
5,6,disc_inst_05,5,5,173.939593,37.264800,35.257607,15.404191,168.893175,"[5, 22]",False,0.7664,
6,7,disc_inst_06,5,5,209.197200,35.257607,NaN,15.429970,168.893175,"[5, 22]",False,0.7660,


## 13. Validación held-out SPIDER

`GATE G` — `PASS / PARTIAL / UNAVAILABLE / FAIL`. SPIDER no fue encontrado localmente en la
Sección 2, así que esta sección documenta el estado `UNAVAILABLE` de forma reproducible (la
lógica de leakage-audit y matching queda implementada y lista para una ejecución futura con
`PFI_POST_E50_SPIDER_ROOT` seteada, pero no se ejecuta sobre datos que no existen).


In [16]:
GATE_G_spider_validation = "UNAVAILABLE"
split_leakage_audit = None
spider_case_metrics = pd.DataFrame()
ground_truth_disc_instances = pd.DataFrame()

if SPIDER_AVAILABLE:
    # Extension point for a future run: inspect SPIDER_ROOT structure (manifests/splits) before
    # assuming any patient/case ID scheme, build split_leakage_audit, derive ground truth disc
    # instances from reference segmentations (not assumed class IDs), and run Hungarian matching
    # against disc_instances_consensus per case. Not implemented in this run because SPIDER is
    # not present locally -- implementing blind (without inspecting real structure) would risk
    # exactly the "assumed numeric IDs" mistake the brief prohibits.
    warnings.append("SPIDER_ROOT was found but structural inspection/leakage-audit is not implemented in this run; GATE G remains UNAVAILABLE until that inspection is done.")
else:
    limitations.append(
        "SPIDER held-out validation is UNAVAILABLE in this run: the dataset was not found "
        "locally (only unrelated lumbar-MRI datasets were present under Downloads). "
        "PFI_POST_E50_SPIDER_ROOT is supported for a future local/Colab/Drive run."
    )

print("GATE_G_spider_validation:", GATE_G_spider_validation)


GATE_G_spider_validation: UNAVAILABLE


## 14. Resultado exploratorio sobre el DICOM real — `DiscLocalization`

Se serializa el resultado en el esquema experimental `pfi.post-e50.disc-localization.v0`.
**No se llama "ground truth"** y ningún nivel se marca `"validated"` sin evidencia externa
independiente -- como máximo `"inferred"` o `"abstain"`.


In [17]:
DISC_LOCALIZATION_SCHEMA_VERSION = "pfi.post-e50.disc-localization.v0"
disc_localizations = []

if len(disc_instances_consensus):
    for _, row in disc_instances_consensus.iterrows():
        level = row.get("level")
        level_status = row.get("level_status", "abstain")
        bbox_candidates = disc_instance_candidates[
            disc_instance_candidates["slice_source"] == row["representative_slice"]
        ]
        bbox_pixel = None
        if len(bbox_candidates):
            closest = bbox_candidates.iloc[0]
            bbox_pixel = closest["bbox"]

        disc_localizations.append({
            "schemaVersion": DISC_LOCALIZATION_SCHEMA_VERSION,
            "studyOpaqueId": STUDY_OPAQUE_ID,
            "instanceId": row["instance_id"],
            "level": level if level else None,
            "levelStatus": level_status if level else "abstain",
            "sourceSeriesRole": "sagittal_t2",
            "representativeSlice": int(row["representative_slice"]),
            "centroidPatientXYZ": row["centroid_patient_xyz"],
            "bboxPixel": bbox_pixel,
            "bboxPhysicalMm": None,
            "supportingSlices": row["supporting_slices"],
            "instanceConfidence": float(row["instance_confidence"]),
            "levelNamingConfidence": float(LEVEL_NAMING_CONFIDENCE),
            "warnings": [],
        })

GATE_H_real_dicom_localization = "FAIL"
if len(disc_localizations) and GATE_E_anatomical_ordering == "PASS":
    GATE_H_real_dicom_localization = "PASS" if GATE_F_absolute_level_naming in ("PASS", "PARTIAL") else "PARTIAL"
elif len(disc_localizations):
    GATE_H_real_dicom_localization = "PARTIAL"

print("GATE_H_real_dicom_localization:", GATE_H_real_dicom_localization)
print(f"Disc instances detected: {len(disc_instances_consensus)}")
print(f"Levels assigned (inferred, NOT validated): {levels_assigned}")
json.dumps(disc_localizations, indent=2, default=str)  # round-trip serialization check


GATE_H_real_dicom_localization: PASS
Disc instances detected: 7
Levels assigned (inferred, NOT validated): 0


'[\n  {\n    "schemaVersion": "pfi.post-e50.disc-localization.v0",\n    "studyOpaqueId": "206aa67ee4e6",\n    "instanceId": "disc_inst_00",\n    "level": null,\n    "levelStatus": "abstain",\n    "sourceSeriesRole": "sagittal_t2",\n    "representativeSlice": 5,\n    "centroidPatientXYZ": [\n      -2.259055810919259,\n      42.6881095392793,\n      176.78101473441808\n    ],\n    "bboxPixel": [\n      4,\n      9,\n      109,\n      131\n    ],\n    "bboxPhysicalMm": null,\n    "supportingSlices": [\n      3,\n      4,\n      5,\n      6\n    ],\n    "instanceConfidence": 0.8695,\n    "levelNamingConfidence": 0.4403,\n    "warnings": []\n  },\n  {\n    "schemaVersion": "pfi.post-e50.disc-localization.v0",\n    "studyOpaqueId": "206aa67ee4e6",\n    "instanceId": "disc_inst_01",\n    "level": null,\n    "levelStatus": "abstain",\n    "sourceSeriesRole": "sagittal_t2",\n    "representativeSlice": 5,\n    "centroidPatientXYZ": [\n      -2.1153143112739308,\n      37.149452398238026,\n      

## 15. Visualizaciones

SPIDER no disponible en esta corrida → las figuras #7 (centroid error por level) y #8 (confusion
matrix) se omiten explícitamente (no hay ground truth) en vez de simularlas.


In [18]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


if MODEL_AVAILABLE and REAL_DICOM_USED and BEST_SLICE_INDEX is not None:
    best_payload = slice_predictions[BEST_SLICE_INDEX]
    best_ds = best_payload["ds"]
    best_entry = next(inst.entry_name for _, inst in slice_ordering[sag_t2_ids[0]] if inst.dataset is best_ds)
    best_native = load_pixel_array_for_entry(DICOM_SOURCE_PATH, best_entry)
    best_prepared = resize_image(best_native, tuple(sagittal_runtime_meta["targetSize"]))
    best_prediction = best_payload["prediction"]

    # 1. Sagittal T2 + segmentation overlay
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(best_prepared, cmap="gray")
    overlay = np.ma.masked_where(best_prediction == 0, best_prediction)
    ax.imshow(overlay, cmap="viridis", alpha=0.45)
    ax.set_title(f"Sagittal T2 slice {BEST_SLICE_INDEX} + segmentation overlay")
    plt.tight_layout(); fig.savefig(FIGURES_DIR / "sagittal_segmentation_overlay.png", dpi=120); plt.close(fig)

    # 2. Detected disc connected components
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(best_prepared, cmap="gray")
    disc_mask_best = best_prediction == disc_class_id if disc_class_id is not None else np.zeros_like(best_prediction, dtype=bool)
    ax.imshow(np.ma.masked_where(~disc_mask_best, disc_mask_best), cmap="autumn", alpha=0.6)
    if len(disc_instance_candidates):
        for _, row in disc_instance_candidates[disc_instance_candidates["slice_source"] == BEST_SLICE_INDEX].iterrows():
            r, c = row["centroid_row_col"]
            ax.scatter([c], [r], c="cyan", marker="+", s=100)
    ax.set_title("Detected disc connected components (best slice)")
    plt.tight_layout(); fig.savefig(FIGURES_DIR / "disc_connected_components.png", dpi=120); plt.close(fig)

    # 3. Multi-slice consensus (centroids across supporting slices, patient X-Z projection)
    fig, ax = plt.subplots(figsize=(6, 6))
    if len(disc_instances_consensus):
        for _, row in disc_instances_consensus.iterrows():
            xyz = np.array(row["centroid_patient_xyz"])
            ax.scatter([xyz[0]], [xyz[2]], s=100, label=f"{row['instance_id']} (n={row['supporting_slice_count']})")
    ax.set_xlabel("Patient X (mm)"); ax.set_ylabel("Patient Z (mm)")
    ax.set_title("Multi-slice disc instance consensus")
    ax.legend(fontsize=6)
    plt.tight_layout(); fig.savefig(FIGURES_DIR / "multi_slice_consensus.png", dpi=120); plt.close(fig)

    # 4. Spine axis + ordered disc centroids
    fig, ax = plt.subplots(figsize=(6, 6))
    if len(disc_instances_consensus) and "position_along_spine_mm" in disc_instances_consensus.columns:
        ordered_df = disc_instances_consensus.sort_values("ordinal_cranial_to_caudal")
        xs = [xyz[0] for xyz in ordered_df["centroid_patient_xyz"]]
        zs = [xyz[2] for xyz in ordered_df["centroid_patient_xyz"]]
        ax.plot(xs, zs, marker="o", color="gray")
        for i, (_, row) in enumerate(ordered_df.iterrows()):
            xyz = row["centroid_patient_xyz"]
            ax.annotate(str(row["ordinal_cranial_to_caudal"]), (xyz[0], xyz[2]))
    ax.set_xlabel("Patient X (mm)"); ax.set_ylabel("Patient Z (mm)")
    ax.set_title("Spine axis ordering (1=most cranial)")
    plt.tight_layout(); fig.savefig(FIGURES_DIR / "spine_axis_ordering.png", dpi=120); plt.close(fig)

    # 5. Predicted levels over sagittal T2
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(best_prepared, cmap="gray")
    ax.imshow(np.ma.masked_where(~disc_mask_best, disc_mask_best), cmap="autumn", alpha=0.5)
    if len(disc_instances_consensus):
        for _, row in disc_instances_consensus.iterrows():
            if row["representative_slice"] != BEST_SLICE_INDEX or not row.get("level"):
                continue
            candidates_here = disc_instance_candidates[disc_instance_candidates["slice_source"] == BEST_SLICE_INDEX]
            if len(candidates_here):
                r, c = candidates_here.iloc[0]["centroid_row_col"]
                ax.annotate(f"{row['level']} ({row['level_status']})", (c, r), color="yellow", fontsize=7)
    ax.set_title("Predicted levels (INFERRED, not validated)")
    plt.tight_layout(); fig.savefig(FIGURES_DIR / "predicted_levels.png", dpi=120); plt.close(fig)

    # 6. Confidence per disc instance
    fig, ax = plt.subplots(figsize=(7, 4))
    if len(disc_instances_consensus):
        ax.bar(disc_instances_consensus["instance_id"], disc_instances_consensus["instance_confidence"])
        ax.set_ylabel("instance_confidence (not a probability)")
        ax.set_title("Confidence per disc instance")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout(); fig.savefig(FIGURES_DIR / "confidence_per_instance.png", dpi=120); plt.close(fig)

    print("Figures 1-6 written. Figures 7-8 (SPIDER ground-truth comparisons) skipped: SPIDER UNAVAILABLE.")
else:
    print("Skipped: model/DICOM unavailable.")


Figures 1-6 written. Figures 7-8 (SPIDER ground-truth comparisons) skipped: SPIDER UNAVAILABLE.


## 16. Quality gates A–I


In [19]:
gate_status: dict[str, str] = {
    "GATE_A_checkpoint_identity": GATE_A_checkpoint_identity,
    "GATE_B_inference_executable": GATE_B_inference_executable,
    "GATE_C_disc_instance_extraction": GATE_C_disc_instance_extraction,
    "GATE_D_multi_slice_consensus": GATE_D_multi_slice_consensus,
    "GATE_E_anatomical_ordering": GATE_E_anatomical_ordering,
    "GATE_F_absolute_level_naming": GATE_F_absolute_level_naming,
    "GATE_G_spider_validation": GATE_G_spider_validation,
    "GATE_H_real_dicom_localization": GATE_H_real_dicom_localization,
    "GATE_I_privacy": None,
}
gates_df = pd.DataFrame(sorted(gate_status.items()), columns=["gate", "status"])
gates_df


,gate,status
0,GATE_A_checkpoint_identity,PASS
1,GATE_B_inference_executable,PASS
2,GATE_C_disc_instance_extraction,PASS
3,GATE_D_multi_slice_consensus,PASS
4,GATE_E_anatomical_ordering,PASS
5,GATE_F_absolute_level_naming,PARTIAL
6,GATE_G_spider_validation,UNAVAILABLE
7,GATE_H_real_dicom_localization,PASS
8,GATE_I_privacy,None


## 17. Privacy audit (GATE I)


In [20]:
forbidden_values = set()
if REAL_DICOM_USED:
    for inst in instances:
        ds = inst.dataset
        for field_name in FORBIDDEN_IDENTIFIER_FIELDS:
            value = getattr(ds, field_name, None)
            if value:
                forbidden_values.add(str(value))
safe_write_text._forbidden_values = forbidden_values

candidate_outputs = {
    "sagittal_slice_inventory": sagittal_slice_inventory.to_csv(index=False) if len(sagittal_slice_inventory) else "",
    "disc_instance_candidates": disc_instance_candidates.to_csv(index=False) if len(disc_instance_candidates) else "",
    "disc_instances_consensus": disc_instances_consensus.to_csv(index=False) if len(disc_instances_consensus) else "",
    "disc_localizations": json.dumps(disc_localizations, default=str),
    "ordered_disc_instance_profile": ordered_disc_instance_profile.to_csv(index=False) if len(ordered_disc_instance_profile) else "",
}

privacy_findings = []
for name, text in candidate_outputs.items():
    for value in forbidden_values:
        if value in text:
            privacy_findings.append(f"{name} contains a raw forbidden identifier value")
    if "C:\\Users\\" in text or "/Users/" in text:
        privacy_findings.append(f"{name} contains a local filesystem path")

GATE_I_PRIVACY_PASS = len(privacy_findings) == 0
gate_status["GATE_I_privacy"] = "PASS" if GATE_I_PRIVACY_PASS else "FAIL"
gates_df.loc[gates_df["gate"] == "GATE_I_privacy", "status"] = gate_status["GATE_I_privacy"]

print("Privacy findings:", privacy_findings if privacy_findings else "(none)")
print("GATE_I_privacy:", gate_status["GATE_I_privacy"])
if not GATE_I_PRIVACY_PASS:
    warnings.append(f"Privacy audit found issues: {privacy_findings}")

print(gates_df)


Privacy findings: (none)
GATE_I_privacy: PASS
                              gate       status
0       GATE_A_checkpoint_identity         PASS
1      GATE_B_inference_executable         PASS
2  GATE_C_disc_instance_extraction         PASS
3     GATE_D_multi_slice_consensus         PASS
4       GATE_E_anatomical_ordering         PASS
5     GATE_F_absolute_level_naming      PARTIAL
6         GATE_G_spider_validation  UNAVAILABLE
7   GATE_H_real_dicom_localization         PASS
8                   GATE_I_privacy         PASS


## 18. Decisión general

Estados posibles: `SAGITTAL_LEVEL_LOCALIZATION_BASELINE_ESTABLISHED`,
`SAGITTAL_LEVEL_LOCALIZATION_VALIDATED_ON_HELDOUT`, `PARTIAL`, `BLOCKED`.
`VALIDATED_ON_HELDOUT` exige SPIDER held-out real + leakage audit PASS + ground truth + métricas
por nivel + pipeline reproducible -- ninguna de estas condiciones se cumple en esta corrida
(SPIDER `UNAVAILABLE`), así que **`VALIDATED_ON_HELDOUT` está descartado de antemano**, sin
importar qué tan bien funcione el DICOM real.


In [21]:
def overall_gate_status(statuses: list[str]) -> str:
    if any(s == "FAIL" for s in statuses):
        return "FAIL"
    if any(s in ("PARTIAL", "UNAVAILABLE") for s in statuses):
        return "PARTIAL"
    return "PASS"


QUALITY_GATE_OVERALL = overall_gate_status(list(gate_status.values()))

core_gates_ok = all(gate_status[g] == "PASS" for g in ["GATE_A_checkpoint_identity", "GATE_B_inference_executable", "GATE_I_privacy"])

if GATE_G_spider_validation == "PASS":
    decision = "SAGITTAL_LEVEL_LOCALIZATION_VALIDATED_ON_HELDOUT"
elif core_gates_ok and gate_status["GATE_C_disc_instance_extraction"] in ("PASS", "PARTIAL") and gate_status["GATE_E_anatomical_ordering"] in ("PASS", "PARTIAL"):
    decision = "SAGITTAL_LEVEL_LOCALIZATION_BASELINE_ESTABLISHED" if QUALITY_GATE_OVERALL != "FAIL" else "PARTIAL"
elif core_gates_ok:
    decision = "PARTIAL"
else:
    decision = "BLOCKED"

blocking_gates = [g for g, s in gate_status.items() if s not in ("PASS",)]

# Notebook 67B needs to know WHICH centroids correspond to L1-L2...L5-S1. Without a resolved
# target lumbar window / absolute level naming, pairing axial clusters to "levels" would be
# pairing them to unlabeled instances -- not meaningful yet. So this depends on GATE F, not just
# on instance detection/ordering quality.
READY_FOR_AXIAL_CLUSTER_PAIRING = bool(
    target_lumbar_window_status == "RESOLVED" and absolute_level_status not in ("abstain",)
)
READY_FOR_SPIDER_HELDOUT_VALIDATION = bool(
    gate_status["GATE_B_inference_executable"] == "PASS"
    and gate_status["GATE_C_disc_instance_extraction"] in ("PASS", "PARTIAL")
    and gate_status["GATE_D_multi_slice_consensus"] in ("PASS", "PARTIAL")
    and gate_status["GATE_E_anatomical_ordering"] in ("PASS", "PARTIAL")
)
READY_FOR_ABSOLUTE_LEVEL_ANCHOR_RESEARCH = READY_FOR_SPIDER_HELDOUT_VALIDATION
BLOCKING_REQUIREMENT = "ABSOLUTE_LEVEL_ANCHOR_VALIDATION" if not READY_FOR_AXIAL_CLUSTER_PAIRING else None

print("QUALITY_GATE_OVERALL:", QUALITY_GATE_OVERALL)
print("Decision:", decision)
print("Blocking gates:", blocking_gates)
print("ready_for_axial_cluster_pairing:", READY_FOR_AXIAL_CLUSTER_PAIRING)
print("ready_for_spider_heldout_validation:", READY_FOR_SPIDER_HELDOUT_VALIDATION)
print("ready_for_absolute_level_anchor_research:", READY_FOR_ABSOLUTE_LEVEL_ANCHOR_RESEARCH)
print("blocking_requirement:", BLOCKING_REQUIREMENT)
print("Note: multiplanar cross-frame registration is NOT required for these flags (that is what 67B investigates, once levels are known).")


QUALITY_GATE_OVERALL: PARTIAL
Decision: SAGITTAL_LEVEL_LOCALIZATION_BASELINE_ESTABLISHED
Blocking gates: ['GATE_F_absolute_level_naming', 'GATE_G_spider_validation']
ready_for_axial_cluster_pairing: False
ready_for_spider_heldout_validation: True
ready_for_absolute_level_anchor_research: True
blocking_requirement: ABSOLUTE_LEVEL_ANCHOR_VALIDATION
Note: multiplanar cross-frame registration is NOT required for these flags (that is what 67B investigates, once levels are known).


## 19. ¿Se necesita Colab ahora?

Con un solo estudio DICOM real (12 slices sagitales) y SPIDER no disponible, el costo
computacional de esta corrida es trivial en CPU. Se documenta la lógica de estimación para una
futura corrida con SPIDER, sin ejecutar ningún batch grande.


In [22]:
EXECUTION_SECONDS_SO_FAR = time.time() - EXECUTION_START
SPIDER_CASE_COUNT_KNOWN = 0  # SPIDER unavailable in this run

COLAB_BATCH_VALIDATION_RECOMMENDED = False
colab_reason = "N/A -- SPIDER unavailable in this run; only a single real-DICOM sagittal series (12 slices) was processed on CPU."
if SPIDER_CASE_COUNT_KNOWN > 50:
    COLAB_BATCH_VALIDATION_RECOMMENDED = True
    colab_reason = f"{SPIDER_CASE_COUNT_KNOWN} SPIDER cases would require batch inference; recommend validating on a handful of held-out cases first, then moving full batch to Colab/GPU."

print("COLAB_BATCH_VALIDATION_RECOMMENDED:", COLAB_BATCH_VALIDATION_RECOMMENDED)
print("Reason:", colab_reason)
print(f"Execution so far: {EXECUTION_SECONDS_SO_FAR:.1f}s (CPU, single study)")


COLAB_BATCH_VALIDATION_RECOMMENDED: False
Reason: N/A -- SPIDER unavailable in this run; only a single real-DICOM sagittal series (12 slices) was processed on CPU.
Execution so far: 4.1s (CPU, single study)


## 20. Artefactos de salida

Todo se escribe exclusivamente dentro de `artifacts/post_e50/level_localization/` y
`reports/post_e50/`. Sin DICOM, sin masks volumétricas gigantes versionadas.


In [23]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)


safe_write_text(LOC_DIR / "localization_environment.json", json.dumps(model_environment, indent=2, default=_json_default, ensure_ascii=False))

if len(sagittal_slice_inventory):
    sagittal_slice_inventory.to_csv(LOC_DIR / "sagittal_slice_inventory.csv", index=False)
if len(disc_instance_candidates):
    disc_instance_candidates.to_csv(LOC_DIR / "disc_instance_candidates.csv", index=False)
if len(disc_instances_consensus):
    disc_instances_consensus.to_csv(LOC_DIR / "disc_instances_consensus.csv", index=False)
if len(vertebral_instance_candidates):
    vertebral_instance_candidates.to_csv(LOC_DIR / "vertebral_instance_candidates.csv", index=False)
if len(ordered_disc_instance_profile):
    ordered_disc_instance_profile.to_csv(LOC_DIR / "ordered_disc_instance_profile.csv", index=False)

safe_write_text(LOC_DIR / "disc_localizations.json", json.dumps(disc_localizations, indent=2, default=_json_default, ensure_ascii=False))

level_localization_metrics = {
    "generated_at": GENERATED_AT,
    "study_opaque_id": STUDY_OPAQUE_ID,
    "disc_instances_detected": int(len(disc_instances_consensus)),
    "ordered_instances": int(len(disc_instances_consensus)) if "position_along_spine_mm" in disc_instances_consensus.columns else 0,
    "target_lumbar_window_status": target_lumbar_window_status,
    "levels_assigned": int(levels_assigned),
    "absolute_level_status": absolute_level_status,
    "abstain_primary_reason": abstain_primary_reason,
    "level_naming_confidence": float(LEVEL_NAMING_CONFIDENCE),
    "anchor_quality_gates": anchor_quality_gates,
    "ground_truth_available": False,
    "note": "No held-out ground truth in this run (SPIDER UNAVAILABLE); no instance-detection or level-naming accuracy metrics computed against ground truth. instance_count in anchor_quality_gates is auxiliary information, not the cause of abstention.",
}
safe_write_text(LOC_DIR / "level_localization_metrics.json", json.dumps(level_localization_metrics, indent=2, default=_json_default, ensure_ascii=False))

safe_write_text(LOC_DIR / "localization_quality_gates.json", json.dumps({"gates": gate_status, "overall_status": QUALITY_GATE_OVERALL, "warnings": warnings}, indent=2, ensure_ascii=False))

if SPIDER_AVAILABLE:
    safe_write_text(LOC_DIR / "split_leakage_audit.json", json.dumps({"status": "not_implemented_pending_structure_inspection"}, indent=2))

localization_summary = {
    "generated_at": GENERATED_AT,
    "git_branch": GIT_BRANCH,
    "git_commit": GIT_COMMIT,
    "model": {"model_id": MODEL_ID, "checkpoint_sha256": checkpoint_sha256},
    "real_dicom_used": REAL_DICOM_USED,
    "study_opaque_id": STUDY_OPAQUE_ID,
    "spider_available": SPIDER_AVAILABLE,
    "disc_instances_detected": int(len(disc_instances_consensus)),
    "target_lumbar_window_status": target_lumbar_window_status,
    "levels_assigned": int(levels_assigned),
    "absolute_level_status": absolute_level_status,
    "abstain_primary_reason": abstain_primary_reason,
    "gates": gate_status,
    "quality_gate_overall": QUALITY_GATE_OVERALL,
    "decision": decision,
    "ready_for_axial_cluster_pairing": READY_FOR_AXIAL_CLUSTER_PAIRING,
    "ready_for_spider_heldout_validation": READY_FOR_SPIDER_HELDOUT_VALIDATION,
    "ready_for_absolute_level_anchor_research": READY_FOR_ABSOLUTE_LEVEL_ANCHOR_RESEARCH,
    "blocking_requirement": BLOCKING_REQUIREMENT,
    "blocking_gates": blocking_gates,
    "colab_batch_validation_recommended": COLAB_BATCH_VALIDATION_RECOMMENDED,
    "colab_reason": colab_reason,
    "automatic_disc_localization_validated_changed": False,
    "warnings": warnings,
    "limitations": limitations,
}
safe_write_text(LOC_DIR / "localization_summary.json", json.dumps(localization_summary, indent=2, default=_json_default, ensure_ascii=False))

print("Written:")
for f in sorted(LOC_DIR.rglob("*")):
    if f.is_file():
        print(" -", f.relative_to(REPO_ROOT))


Written:
 - artifacts\post_e50\level_localization\disc_instance_candidates.csv
 - artifacts\post_e50\level_localization\disc_instances_consensus.csv
 - artifacts\post_e50\level_localization\disc_localizations.json
 - artifacts\post_e50\level_localization\figures\confidence_per_instance.png
 - artifacts\post_e50\level_localization\figures\disc_connected_components.png
 - artifacts\post_e50\level_localization\figures\multi_slice_consensus.png
 - artifacts\post_e50\level_localization\figures\predicted_levels.png
 - artifacts\post_e50\level_localization\figures\sagittal_segmentation_overlay.png
 - artifacts\post_e50\level_localization\figures\spine_axis_ordering.png
 - artifacts\post_e50\level_localization\level_localization_metrics.json
 - artifacts\post_e50\level_localization\localization_environment.json
 - artifacts\post_e50\level_localization\localization_quality_gates.json
 - artifacts\post_e50\level_localization\localization_summary.json
 - artifacts\post_e50\level_localization\orde

## 21. Limitaciones (agregado)


In [24]:
limitations.extend([
    "Non-diagnostic research output; not a medical device.",
    "Sagittal-only in this notebook: Axial T2 and cross-frame registration were explicitly out of scope.",
    "The sagittal_spider segmentation model may exhibit domain shift on this external real-world DICOM study (trained on SPIDER).",
    "disc_group is not a native level-labelled output of the model -- level naming is a downstream heuristic, not a model capability.",
    "Absolute level naming requires a confident anatomical anchor; this pipeline has no dedicated sacral/S1 class, so the inferior anchor is never treated as fully validated -- only as a heuristic candidate, marked 'inferred'.",
    "Transitional or atypical anatomy, partial coverage, or segmentation fragmentation trigger ABSTAIN rather than a forced 5-level assignment.",
    "The real DICOM study provides NO ground truth: any levels produced are labeled INFERRED, never VALIDATED.",
    "A strong 'validated' claim requires SPIDER held-out validation with a real, leakage-audited test split -- unavailable in this run.",
    "No axial pairing was attempted; readiness for Notebook 67B reflects only sagittal-side stability.",
    "AUTOMATIC_DISC_LOCALIZATION_VALIDATED was NOT changed by this notebook.",
])
for w in warnings:
    if w not in limitations:
        limitations.append(w)
for item in limitations:
    print("-", item)


- SPIDER held-out validation is UNAVAILABLE in this run: the dataset was not found locally (only unrelated lumbar-MRI datasets were present under Downloads). PFI_POST_E50_SPIDER_ROOT is supported for a future local/Colab/Drive run.
- Non-diagnostic research output; not a medical device.
- Sagittal-only in this notebook: Axial T2 and cross-frame registration were explicitly out of scope.
- The sagittal_spider segmentation model may exhibit domain shift on this external real-world DICOM study (trained on SPIDER).
- disc_group is not a native level-labelled output of the model -- level naming is a downstream heuristic, not a model capability.
- Absolute level naming requires a confident anatomical anchor; this pipeline has no dedicated sacral/S1 class, so the inferior anchor is never treated as fully validated -- only as a heuristic candidate, marked 'inferred'.
- Transitional or atypical anatomy, partial coverage, or segmentation fragmentation trigger ABSTAIN rather than a forced 5-level

## 22. Reporte (Markdown)


In [25]:
report_lines = []
report_lines.append("# Post-E50 Sagittal Level Localization")
report_lines.append("")
report_lines.append("## Objective")
report_lines.append("")
report_lines.append(
    "Build a reproducible pipeline from Sagittal T2 through the existing sagittal_spider "
    "segmentation to individual disc instances, anatomical cranio-caudal ordering, and (attempted) "
    "absolute lumbar level naming with explicit abstention when evidence is insufficient."
)
report_lines.append("")
report_lines.append("## Scope")
report_lines.append("")
report_lines.append(
    "Sagittal-only (Sagittal T2 primary, Sagittal T1 as secondary evidence only). No Axial T2, "
    "no cross-frame registration, no Notebook 67B, no training, no productive code changes, no "
    "change to AUTOMATIC_DISC_LOCALIZATION_VALIDATED."
)
report_lines.append("")
report_lines.append("## Frozen model")
report_lines.append("")
report_lines.append(f"```json\n{json.dumps({k: v for k, v in model_environment.items() if k != 'class_names'}, indent=2, default=str)}\n```")
report_lines.append(f"- class_names: `{model_environment['class_names']}`")
report_lines.append(f"- GATE A (checkpoint identity): `{gate_status['GATE_A_checkpoint_identity']}`")
report_lines.append("")
report_lines.append("## Data sources")
report_lines.append("")
report_lines.append(f"- Real DICOM used: `{REAL_DICOM_USED}` (discovery method: `{DICOM_SOURCE_METHOD.split(':')[0] if REAL_DICOM_USED else DICOM_SOURCE_METHOD}`)")
report_lines.append(f"- ZIP SHA-256 matches historical expected value: `{dicom_zip_sha256 == EXPECTED_STUDY_ZIP_SHA256 if dicom_zip_sha256 else None}`")
report_lines.append(f"- SPIDER available locally: `{SPIDER_AVAILABLE}` (discovery method: `{SPIDER_DISCOVERY_METHOD}`)")
report_lines.append("")
report_lines.append("## Leakage audit")
report_lines.append("")
report_lines.append("Not applicable in this run: SPIDER is UNAVAILABLE, so no train/val/test split leakage audit was performed. Logic reserved for a future run with `PFI_POST_E50_SPIDER_ROOT` set.")
report_lines.append("")
report_lines.append("## Sagittal inference")
report_lines.append("")
report_lines.append(f"- GATE B (inference executable): `{gate_status['GATE_B_inference_executable']}`")
report_lines.append(f"- Slices processed: `{len(sagittal_slice_inventory)}`")
report_lines.append(f"- Best slice (by `sagittal_slice_quality_score`): `{BEST_SLICE_INDEX}`")
report_lines.append(f"- Top-K useful slices: `{top_k_slices}`")
report_lines.append("")
report_lines.append(sagittal_slice_inventory.to_markdown(index=False) if len(sagittal_slice_inventory) else "_No slice inventory available._")
report_lines.append("")
report_lines.append("## Disc instance extraction")
report_lines.append("")
report_lines.append(f"- GATE C: `{gate_status['GATE_C_disc_instance_extraction']}`")
report_lines.append(f"- Raw components extracted: `{len(disc_instance_candidates)}`")
report_lines.append("")
report_lines.append("## Multi-slice consensus")
report_lines.append("")
report_lines.append(f"- GATE D: `{gate_status['GATE_D_multi_slice_consensus']}`")
report_lines.append(disc_instances_consensus.to_markdown(index=False) if len(disc_instances_consensus) else "_No consensus instances._")
report_lines.append("")
report_lines.append("## Spine axis")
report_lines.append("")
report_lines.append(f"- GATE E (anatomical ordering): `{gate_status['GATE_E_anatomical_ordering']}`")
report_lines.append(f"- spine_axis_vector (points caudally, patient LPS): `{spine_axis_vector.tolist() if spine_axis_vector is not None else None}`")
report_lines.append(f"- Vertebral instances separable: `{VERTEBRA_INSTANCES_SEPARABLE}`")
report_lines.append("")
report_lines.append("## Absolute level naming")
report_lines.append("")
report_lines.append(f"- GATE F: `{gate_status['GATE_F_absolute_level_naming']}`")
report_lines.append(f"- target_lumbar_window_status: `{target_lumbar_window_status}`")
report_lines.append(f"- anchor_quality_gates (auxiliary information only, never sufficient cause): `{anchor_quality_gates}`")
report_lines.append(f"- absolute_level_status: `{absolute_level_status}`")
report_lines.append(f"- abstain_primary_reason: `{abstain_primary_reason}`")
report_lines.append(f"- levels_assigned: `{levels_assigned}`")
report_lines.append("")
report_lines.append("## Abstention logic")
report_lines.append("")
report_lines.append(
    "Three separate tasks: (1) disc instance detection, (2) target lumbar level selection "
    "(which of the detected instances are L1-L2...L5-S1), (3) absolute level naming. A "
    "field-of-view containing more or fewer than 5 candidates is NOT by itself evidence that "
    "instance detection is wrong -- the sagittal FOV can legitimately include discs outside the "
    "lumbar target window. Absolute naming abstains because `target_lumbar_window_status = "
    f"{target_lumbar_window_status}`: no genuine anatomical anchor (SPIDER reference, a "
    "dedicated sacral/S1 class, a validated vertebral-instance anchor, or dataset metadata) is "
    "available in this run. The primary cause is `ABSOLUTE_ANATOMICAL_ANCHOR_UNAVAILABLE`, not "
    "the instance count, which is recorded only as auxiliary information."
)
report_lines.append("")
report_lines.append("## SPIDER held-out validation")
report_lines.append("")
report_lines.append(f"- GATE G: `{gate_status['GATE_G_spider_validation']}` -- dataset not found locally in this run.")
report_lines.append("")
report_lines.append("## Real DICOM exploratory result")
report_lines.append("")
report_lines.append(f"- GATE H: `{gate_status['GATE_H_real_dicom_localization']}`")
report_lines.append("- This is NOT ground truth. No levels were assigned; all instances are `abstain`.")
report_lines.append("")
report_lines.append("### `ordered_disc_instance_profile` (cranial -> caudal, no levels assigned)")
report_lines.append("")
report_lines.append(ordered_disc_instance_profile.to_markdown(index=False) if len(ordered_disc_instance_profile) else "_Not available._")
report_lines.append("")
report_lines.append(json.dumps(disc_localizations, indent=2, default=str))
report_lines.append("")
report_lines.append("## Quality gates")
report_lines.append("")
report_lines.append(gates_df.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Results")
report_lines.append("")
report_lines.append(f"- Overall quality gate: `{QUALITY_GATE_OVERALL}`")
report_lines.append(f"- Decision: `{decision}`")
report_lines.append(f"- ready_for_axial_cluster_pairing: `{READY_FOR_AXIAL_CLUSTER_PAIRING}`")
report_lines.append(f"- ready_for_spider_heldout_validation: `{READY_FOR_SPIDER_HELDOUT_VALIDATION}`")
report_lines.append(f"- ready_for_absolute_level_anchor_research: `{READY_FOR_ABSOLUTE_LEVEL_ANCHOR_RESEARCH}`")
report_lines.append(f"- blocking_requirement: `{BLOCKING_REQUIREMENT}`")
report_lines.append("")
report_lines.append("## Failure cases")
report_lines.append("")
report_lines.append(f"- Absolute level naming ABSTAINED for all {len(disc_instances_consensus)} instances: `{abstain_primary_reason}`. "
                     "This is not interpreted as an instance-detection failure.")
report_lines.append("")
report_lines.append("## Limitations")
report_lines.append("")
for item in limitations:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## What Notebook 67 proves")
report_lines.append("")
report_lines.append(
    "> The frozen sagittal_spider checkpoint can be run across the real study's Sagittal T2; "
    "disc_group can be transformed into instance candidates; those candidates can be "
    "consolidated across slices; the consolidated instances can be ordered reproducibly in "
    "patient-space; and reproducible centroids/ROIs are generated for each. Absolute lumbar "
    "level naming remains unresolved -- it is NOT claimed here."
)
report_lines.append("")
report_lines.append("## What Notebook 67 does not prove")
report_lines.append("")
for item in [
    "no clinical validation",
    "automatic lumbar level localization is NOT validated -- only instance detection, consensus, and ordering are demonstrated",
    "no resolved target lumbar window: which detected instances fall in L1-L2...L5-S1 is unknown",
    "no dedicated sacral/S1 anchor -- absolute level naming has no anatomical anchor to rely on for this model/study",
    "no proof of generalization beyond this single real study",
    "no axial or multiplanar correspondence (out of scope, see Notebook 66B)",
    "AUTOMATIC_DISC_LOCALIZATION_VALIDATED remains False and unchanged",
]:
    report_lines.append(f"- {item}")
report_lines.append("")
report_lines.append("## Ready for Notebook 67B?")
report_lines.append("")
report_lines.append(f"`ready_for_axial_cluster_pairing = {READY_FOR_AXIAL_CLUSTER_PAIRING}` -- Notebook 67B needs to know which "
                     "centroids correspond to L1-L2...L5-S1 to pair them against axial clusters, and no validated absolute "
                     "level naming exists yet.")
report_lines.append("")
report_lines.append("## Recommended next experiment")
report_lines.append("")
report_lines.append(
    "**`67A_postE50_spider_level_anchor_validation.ipynb`** -- use SPIDER held-out data to: "
    "(1) validate instance detection against real ground truth; (2) measure centroid error; "
    "(3) study how many discs appear across different fields of view; (4) learn/validate how to "
    "select the target lumbar window; (5) validate absolute level naming L1-L2...L5-S1; "
    "(6) measure abstention correctly. Only after 67A resolves the anchor question should "
    "**`67B_postE50_axial_cluster_level_pairing.ipynb`** attempt to pair axial clusters to named "
    "levels."
)
report_lines.append("")

report_text = "\n".join(report_lines)
safe_write_text(REPORT_DIR / "post_e50_level_localization_report.md", report_text)
print(f"Report written: {(REPORT_DIR / 'post_e50_level_localization_report.md').relative_to(REPO_ROOT)}")


Report written: reports\post_e50\post_e50_level_localization_report.md


## 23. EXPERIMENT STATUS


In [26]:
EXECUTION_SECONDS = time.time() - EXECUTION_START

status_block = f'''EXPERIMENT STATUS

Experiment:
Post-E50 Sagittal Automatic Level Localization

Training performed:
NO

Model modification:
NO

Checkpoint modification:
NO

Frozen source modified:
NO

Branch:
{GIT_BRANCH}

Notebook:
67_postE50_level_localization_v2.ipynb

Sagittal model:
{MODEL_ID}

Checkpoint SHA256:
{checkpoint_sha256}

Real DICOM used:
{"YES" if REAL_DICOM_USED else "NO"}

SPIDER held-out used:
{"YES" if SPIDER_AVAILABLE else "NO"}

Leakage audit:
{"UNAVAILABLE" if not SPIDER_AVAILABLE else "PASS"}

Disc instances detected:
{len(disc_instances_consensus)}

Ordered instances:
{len(disc_instances_consensus) if "position_along_spine_mm" in disc_instances_consensus.columns else 0}

Target lumbar window:
{target_lumbar_window_status}

Absolute levels assigned:
{levels_assigned}

Absolute level naming:
{"ABSTAIN" if absolute_level_status == "abstain" else absolute_level_status.upper()}

SPIDER held-out validation:
{gate_status["GATE_G_spider_validation"]}

Decision:
{decision}

Ready for SPIDER held-out validation:
{"YES" if READY_FOR_SPIDER_HELDOUT_VALIDATION else "NO"}

Ready for absolute level anchor research:
{"YES" if READY_FOR_ABSOLUTE_LEVEL_ANCHOR_RESEARCH else "NO"}

Ready for axial cluster pairing:
{"YES" if READY_FOR_AXIAL_CLUSTER_PAIRING else "NO"}

Blocking requirement:
{BLOCKING_REQUIREMENT if BLOCKING_REQUIREMENT else "[]"}

Ground-truth localization metrics:
UNAVAILABLE

Real DICOM localization:
{gate_status["GATE_H_real_dicom_localization"]}

Privacy:
{gate_status["GATE_I_privacy"]}

Overall quality gate:
{QUALITY_GATE_OVERALL}

Blocking gates:
{blocking_gates if blocking_gates else "[]"}

AUTOMATIC_DISC_LOCALIZATION_VALIDATED changed:
NO

Execution seconds:
{EXECUTION_SECONDS:.1f}

Warnings:
{chr(10).join(warnings) if warnings else "(none)"}
'''

print(status_block)


EXPERIMENT STATUS

Experiment:
Post-E50 Sagittal Automatic Level Localization

Training performed:
NO

Model modification:
NO

Checkpoint modification:
NO

Frozen source modified:
NO

Branch:
research/post-e50-level-localization

Notebook:
67_postE50_level_localization_v2.ipynb

Sagittal model:
sagittal_spider

Checkpoint SHA256:
cf11dcc0ad77a7c787e64a796a2fd7398ef906add461cef4b3d61f1a5238e944

Real DICOM used:
YES

SPIDER held-out used:
NO

Leakage audit:
UNAVAILABLE

Disc instances detected:
7

Ordered instances:
7

Target lumbar window:
AMBIGUOUS

Absolute levels assigned:
0

Absolute level naming:
ABSTAIN

SPIDER held-out validation:
UNAVAILABLE

Decision:
SAGITTAL_LEVEL_LOCALIZATION_BASELINE_ESTABLISHED

Ready for SPIDER held-out validation:
YES

Ready for absolute level anchor research:
YES

Ready for axial cluster pairing:
NO

Blocking requirement:
ABSOLUTE_LEVEL_ANCHOR_VALIDATION

Ground-truth localization metrics:
UNAVAILABLE

Real DICOM localization:
PASS

Privacy:
PASS

Over